# Block 8 — Steering Analysis: Beautiful Plots

Compares three conditions per group (Human / AI / CoT AI):
- **Original scores** — accuracy_score assigned by Llama3.1:8b in the original Ollama evaluation (from HDF5)
- **Model argmax α=0** — reconstructed score from a clean forward pass (no steering hook)
- **Model argmax α=best** — best steering alpha (max bias reduction)
- **Expected score** — probability-weighted mean Σ d·P(d) instead of argmax

Style: Nature-style Plotly violins with Mann-Whitney p-values and FDR correction.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import h5py
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.multitest import multipletests

from src.config import load_config

cfg       = load_config(ROOT / "config/config.yaml")
MODEL_KEY = cfg.primary_model.key

DATA_DIR  = ROOT / cfg.paths["data_dir"]
STEER_DIR = ROOT / cfg.paths["steer_dir"]
TABLE_DIR = ROOT / cfg.paths["table_dir"]

print(f"Model: {MODEL_KEY}")
print(f"Steering dir: {STEER_DIR}")

In [ ]:
# ── Load original Llama evaluation scores (from HDF5 metadata) ──────────────
with h5py.File(DATA_DIR / f"{MODEL_KEY}.h5", "r") as f:
    orig_sources = f["/sample_metadata/response_source"][()].astype(str)
    orig_acc     = f["/sample_metadata/accuracy_score"][()].astype(float)

orig_df = pd.DataFrame({"group": orig_sources, "accuracy_score": orig_acc})
orig_df = orig_df[orig_df["accuracy_score"] >= 0].copy()

print("Original scores (Ollama Llama3.1:8b):")
print(orig_df.groupby("group")["accuracy_score"].describe().round(2))

# ── Load steering sample CSV ─────────────────────────────────────────────────
smp_path = STEER_DIR / f"{MODEL_KEY}_elasticnet_steering_samples.csv"
smp_df   = pd.read_csv(smp_path)
print(f"\nSteering samples: {len(smp_df)} rows")
print(f"  probe_types: {smp_df['probe_type'].unique().tolist()}")
print(f"  dir_kinds:   {smp_df['dir_kind'].unique().tolist()}")
print(f"  steer_layers:{sorted(smp_df['steer_layer'].unique().tolist())}")
print(f"  alphas:      {sorted(smp_df['alpha'].unique().tolist())}")
print(f"  groups:      {smp_df['group'].unique().tolist()}")

In [ ]:
# ── Load aggregate CSV to find best alpha per (probe_type, dir_kind, layer) ──
agg_path = STEER_DIR / f"{MODEL_KEY}_elasticnet_steering.csv"
agg_df   = pd.read_csv(agg_path)

# Baseline bias = bias_delta at alpha=0 for authorship / elasticnet
bl_row = agg_df[
    (agg_df["probe_type"] == "authorship") &
    (agg_df["alpha"] == 0.0) &
    (agg_df["dir_kind"] == "elasticnet")
].iloc[0]
BASELINE_BIAS = bl_row["bias_delta"]
print(f"Baseline bias (α=0, authorship, elasticnet): {BASELINE_BIAS:+.3f}")

def best_alpha(probe_type, dir_kind, steer_layer):
    """Alpha that maximally reduces |bias| for this (probe_type, dir_kind, layer)."""
    sub = agg_df[
        (agg_df["probe_type"] == probe_type) &
        (agg_df["dir_kind"]   == dir_kind) &
        (agg_df["steer_layer"] == steer_layer)
    ].copy()
    if sub.empty:
        return None
    sub["bias_red"] = (BASELINE_BIAS - sub["bias_delta"]).abs()
    return float(sub.loc[sub["bias_red"].idxmax(), "alpha"])

# Print best alphas
for pt in ["authorship", "accuracy"]:
    for dk in ["elasticnet", "logistic"]:
        for sl in sorted(agg_df["steer_layer"].unique()):
            ba = best_alpha(pt, dk, sl)
            if ba is not None:
                print(f"  {pt} | {dk} | L{sl} → best α = {ba}")

In [ ]:
# ── Statistical helpers ────────────────────────────────────────────────────────

def mw_pvalue(a, b):
    """Two-sided Mann-Whitney U p-value. Returns nan if insufficient data."""
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    try:
        _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        return p
    except Exception:
        return np.nan
def wil_pvalue(a, b):
    """Two-sided Mann-Whitney U p-value. Returns nan if insufficient data."""
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    try:
        _, p = stats.wilcoxon(a, b, alternative="two-sided")
        return p
    except Exception:
        return np.nan

def fmt_p(p):
    if np.isnan(p):
        return "ns"
    if p < 0.001:
        return "p<0.001"
    return f"p={p:.3f}"

def add_bracket(fig, x1, x2, y, text, xref, yref):
    """Significance bracket between x1 and x2 at height y — Nature style."""
    kw = dict(xref=xref, yref=yref, line=dict(color="black", width=2))
    fig.add_shape(type="line", x0=x1, y0=y, x1=x2, y1=y, **kw)
    for xv in [x1, x2]:
        fig.add_shape(type="line", x0=xv, y0=y, x1=xv, y1=y - 0.2, **kw)
    fig.add_annotation(
        x=(x1 + x2) / 2, y=y + 0.33,
        text=text, showarrow=False,
        font=dict(size=16, family="Arial", color="black"),
        xref=xref, yref=yref,
    )

def subplot_refs(subplot_num, n_rows, n_cols):
    """Return (xref, yref) strings for a given 1-indexed subplot number."""
    if n_rows == 1 and n_cols == 1:
        return "x", "y"
    if n_rows == 1:
        return ("x" if subplot_num == 1 else f"x{subplot_num}",
                "y" if subplot_num == 1 else f"y{subplot_num}")
    return (f"x{subplot_num}" if subplot_num > 1 else "x",
            f"y{subplot_num}" if subplot_num > 1 else "y")

def get_orig_scores(grp: str) -> np.ndarray:
    """
    Get original Ollama accuracy scores for a group.
    'AI' → merges AI + CoT AI (matches how the steering experiment combines them).
    """
    if grp == "AI":
        return orig_df[orig_df["group"].isin(["AI", "CoT AI"])]["accuracy_score"].dropna().values
    return orig_df[orig_df["group"] == grp]["accuracy_score"].dropna().values

In [ ]:
# ── Core plotting function ───────────────────────────────────────────────────

GROUP_COLORS = {
    "Human":  "rgb(31, 119, 180)",
    "AI":     "rgb(255, 127, 14)",
    "CoT AI": "rgb(44, 160, 44)",
}

BRACKET_HEIGHTS = {
    ("Human", "AI"):     7.5,
    ("Human", "CoT AI"): 9.1,
    ("AI",    "CoT AI"): 8.3,
}

def _violin_figure(conditions, group_data_by_cond, groups, title_text,
                   metric_title, height, width):
    """Shared figure-building logic for all violin plot functions."""
    n_cols = len(conditions)

    # ── Global FDR correction ────────────────────────────────────────────
    pv_infos = []
    for ci in range(n_cols):
        gd = group_data_by_cond[ci]
        for g1, g2 in [("Human", "AI"), ("Human", "CoT AI"), ("AI", "CoT AI")]:
            if len(gd.get(g1, [])) < 2 or len(gd.get(g2, [])) < 2:
                continue
            p = mw_pvalue(gd[g1], gd[g2])
            if not np.isnan(p):
                pv_infos.append({"ci": ci, "g1": g1, "g2": g2, "p_raw": p})

    if pv_infos:
        _, pv_adj, _, _ = multipletests(
            [x["p_raw"] for x in pv_infos], alpha=0.05, method="fdr_bh")
        for i, info in enumerate(pv_infos):
            info["p_adj"] = pv_adj[i]

    pv_lookup = {(x["ci"], x["g1"], x["g2"]): x.get("p_adj", x["p_raw"])
                 for x in pv_infos}

    fig = make_subplots(
        rows=1, cols=n_cols,
        subplot_titles=[c["label"] for c in conditions],
        horizontal_spacing=0.04,
        shared_yaxes=True,
    )

    x_pos        = {g: i for i, g in enumerate(groups)}
    shown_legend = set()

    for ci, cond in enumerate(conditions):
        col = ci + 1
        gd  = group_data_by_cond[ci]

        for grp in groups:
            vals = gd.get(grp, np.array([]))
            if len(vals) == 0:
                continue
            color    = GROUP_COLORS.get(grp, "gray")
            show_leg = grp not in shown_legend
            shown_legend.add(grp)

            fig.add_trace(
                go.Violin(
                    y=vals,
                    name=f"{grp} (n={len(vals)})",
                    x0=x_pos[grp],
                    fillcolor=color,
                    opacity=0.7,
                    box_visible=True,
                    meanline_visible=True,
                    width=0.5,
                    showlegend=show_leg,
                    legendgroup=grp,
                    line=dict(width=2, color="black"),
                    marker=dict(color=color, line=dict(color="black", width=1)),
                    scalemode="width",
                ),
                row=1, col=col,
            )

        xref, yref = subplot_refs(ci + 1, 1, n_cols)
        for (g1, g2), y_h in BRACKET_HEIGHTS.items():
            if g1 not in x_pos or g2 not in x_pos:
                continue
            p = pv_lookup.get((ci, g1, g2)) or pv_lookup.get((ci, g2, g1))
            if p is None:
                continue
            add_bracket(fig, x_pos[g1], x_pos[g2], y_h, fmt_p(p), xref, yref)

    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white",
        plot_bgcolor="white", paper_bgcolor="white",
        title=dict(text=title_text, font=dict(size=22, family="Arial"), y=0.98),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08,
            xanchor="center", x=0.5,
            font=dict(size=22, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=100, b=80, l=100, r=60),
    )

    for ci in range(n_cols):
        col = ci + 1
        yax = "yaxis"  if col == 1 else f"yaxis{col}"
        xax = "xaxis"  if col == 1 else f"xaxis{col}"
        fig.layout[yax].update(
            title=dict(text=metric_title if col == 1 else "",
                       font=dict(size=26, family="Arial", color="black")),
            range=[0, 9.5],
            showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        )
        fig.layout[xax].update(
            ticktext=groups, tickvals=list(range(len(groups))),
            showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        )

    for ann in fig.layout.annotations[:n_cols]:
        ann.font.size = 16; ann.font.family = "Arial"; ann.font.color = "black"

    return fig


def steering_violin_plot(
    probe_type    = "authorship",
    dir_kind      = "elasticnet",
    steer_layer   = 15,
    metric        = "argmax_score",
    groups        = ["Human", "AI"],
    selected_alphas = None,
    show_original = True,
    height = 900,
    width  = None,
):
    """Symmetric steering violin. 'AI' in original panel = AI + CoT AI combined."""
    if selected_alphas is None:
        all_alphas = sorted(smp_df["alpha"].unique())
        ba = best_alpha(probe_type, dir_kind, steer_layer) or 0.0
        selected_alphas = sorted(set([0.0, ba, max(all_alphas)]))

    conditions = []
    if show_original:
        conditions.append({"label": "Original\n(Ollama)", "source": "original", "alpha": None})
    for α in selected_alphas:
        label = f"α={α:.1f}" + (" (baseline)" if α == 0 else "")
        conditions.append({"label": label, "source": "steer", "alpha": α})

    if width is None:
        width = max(900, len(conditions) * len(groups) * 280)

    metric_title = {"argmax_score": "Score (argmax, 1–7)",
                    "expected_score": "Expected score Σ d·P(d)"}.get(metric, metric)

    group_data_by_cond = {}
    for ci, cond in enumerate(conditions):
        gd = {}
        for grp in groups:
            if cond["source"] == "original":
                vals = get_orig_scores(grp)
            else:
                sub = smp_df[
                    (smp_df["probe_type"]  == probe_type) &
                    (smp_df["dir_kind"]    == dir_kind) &
                    (smp_df["steer_layer"] == steer_layer) &
                    (smp_df["alpha"]       == cond["alpha"]) &
                    (smp_df["group"]       == grp)
                ]
                vals = sub[metric].dropna().values if not sub.empty else np.array([])
            gd[grp] = vals
        group_data_by_cond[ci] = gd

    title = f"Steering — {probe_type} | {dir_kind} | L{steer_layer} | {metric}"
    fig = _violin_figure(conditions, group_data_by_cond, groups, title, metric_title, height, width)
    fig.show()
    return fig


def asym_violin_plot(
    probe_type    = "authorship",
    dir_kind      = "elasticnet",
    steer_layer   = 15,
    metric        = "argmax_score",
    groups        = ["Human", "AI"],
    selected_alphas = None,
    show_original = True,
    height = 900,
    width  = None,
):
    """Asymmetric steering violin (AI: h−α·v, Human: h+α·v). 'AI' = AI + CoT AI."""
    if selected_alphas is None:
        all_alphas = sorted(asym_smp_df["alpha"].unique())
        ba = best_alpha_asym(probe_type, dir_kind, steer_layer) or 0.0
        selected_alphas = sorted(set([0.0, ba, max(all_alphas)]))

    conditions = []
    if show_original:
        conditions.append({"label": "Original\n(Ollama)", "source": "original", "alpha": None})
    for α in selected_alphas:
        label = f"α={α:.1f}" + (" (baseline)" if α == 0 else "")
        conditions.append({"label": label, "source": "asym", "alpha": α})

    if width is None:
        width = max(900, len(conditions) * len(groups) * 280)

    metric_title = {"argmax_score": "Score (argmax, 1–7)",
                    "expected_score": "Expected score Σ d·P(d)"}.get(metric, metric)

    group_data_by_cond = {}
    for ci, cond in enumerate(conditions):
        gd = {}
        for grp in groups:
            if cond["source"] == "original":
                vals = get_orig_scores(grp)
            else:
                sub = asym_smp_df[
                    (asym_smp_df["probe_type"]  == probe_type) &
                    (asym_smp_df["dir_kind"]    == dir_kind) &
                    (asym_smp_df["steer_layer"] == steer_layer) &
                    (asym_smp_df["alpha"]       == cond["alpha"]) &
                    (asym_smp_df["group"]       == grp)
                ]
                vals = sub[metric].dropna().values if not sub.empty else np.array([])
            gd[grp] = vals
        group_data_by_cond[ci] = gd

    title = (f"Asymmetric steering — {probe_type} | {dir_kind} | L{steer_layer} | {metric}<br>"
             "<span style='font-size:18px'>AI: h−α·v  |  Human: h+α·v</span>")
    fig = _violin_figure(conditions, group_data_by_cond, groups, title, metric_title, height, width)
    fig.layout.margin.t = 110
    fig.show()
    return fig


print("steering_violin_plot(), asym_violin_plot() defined.")

## 1. Authorship probe — Argmax score (Human vs AI)

In [ ]:
fig = steering_violin_plot(
    probe_type   = "authorship",
    dir_kind     = "elasticnet",
    steer_layer  = 15,
    metric       = "argmax_score",
    groups       = ["Human", "AI"],
    show_original= True,
    height=700, width=1600,
)

## 2. Same with expected score

In [ ]:
fig = steering_violin_plot(
    probe_type   = "authorship",
    dir_kind     = "elasticnet",
    steer_layer  = 15,
    metric       = "expected_score",
    groups       = ["Human", "AI"],
    show_original= True,
    height=700, width=1600,
)

## 3. Compare layers: L2 vs L15 vs L28 (authorship, elasticnet)

In [ ]:
from plotly.subplots import make_subplots

# One row per layer, each showing: original | α=0 | best α | max α
layers_to_show = [2, 15, 28]
metric = "argmax_score"
dir_kind = "elasticnet"
probe_type = "authorship"
groups = ["Human", "AI"]

for sl in layers_to_show:
    ba = best_alpha(probe_type, dir_kind, sl) or 0.0
    ma = sorted(smp_df["alpha"].unique())[-1]
    fig = steering_violin_plot(
        probe_type=probe_type, dir_kind=dir_kind,
        steer_layer=sl, metric=metric,
        groups=groups,
        selected_alphas=sorted(set([0.0, ba, ma])),
        show_original=True,
        height=650, width=1400,
    )
    fig.update_layout(title_text=f"{probe_type} | {dir_kind} | L{sl} | {metric}")
    fig.show()

## 4. Logistic vs Elastic-net direction at L15

In [ ]:
for dk in ["logistic", "elasticnet"]:
    ba = best_alpha("authorship", dk, 15) or 0.0
    ma = sorted(smp_df["alpha"].unique())[-1]
    fig = steering_violin_plot(
        probe_type="authorship", dir_kind=dk, steer_layer=15,
        metric="argmax_score", groups=["Human", "AI"],
        selected_alphas=sorted(set([0.0, ba, ma])),
        show_original=True,
        height=650, width=1400,
    )
    fig.update_layout(title_text=f"authorship | {dk} direction | L15")
    fig.show()

## 5. Summary: Bias delta vs alpha (all layers)

In [ ]:
# Line plot: bias delta vs alpha per (layer, dir_kind) for authorship
sub = agg_df[agg_df["probe_type"] == "authorship"].copy()
layers = sorted(sub["steer_layer"].unique())
dir_kinds = sorted(sub["dir_kind"].unique())

_colors_tab10 = [
    "rgb(31,119,180)","rgb(255,127,14)","rgb(44,160,44)",
    "rgb(214,39,40)",  "rgb(148,103,189)","rgb(140,86,75)",
]
_dash = {"elasticnet": "solid", "logistic": "dash"}

fig = go.Figure()
for li, sl in enumerate(layers):
    for dk in dir_kinds:
        sub2 = sub[(sub["steer_layer"] == sl) & (sub["dir_kind"] == dk)].sort_values("alpha")
        if sub2.empty:
            continue
        bias_red = ((BASELINE_BIAS - sub2["bias_delta"]) / abs(BASELINE_BIAS) * 100
                    if abs(BASELINE_BIAS) > 1e-8 else 0)
        fig.add_trace(go.Scatter(
            x=sub2["alpha"], y=bias_red,
            mode="lines+markers",
            name=f"L{sl} {dk}",
            line=dict(color=_colors_tab10[li % len(_colors_tab10)],
                      dash=_dash.get(dk, "solid"), width=2.5),
            marker=dict(size=6),
        ))

fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.add_hline(y=100, line_dash="dash", line_color="green",
              annotation_text="Full bias elimination", annotation_position="right")
fig.update_layout(
    title="Authorship probe — Bias reduction (%) vs alpha",
    xaxis_title="Alpha",
    yaxis_title="Bias reduction (%)",
    font=dict(size=18, family="Arial"),
    template="plotly_white",
    height=550, width=1000,
    legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
)
fig.show()

In [ ]:
# ── Mean ± std of argmax score per group across alpha values ─────────────────
# Groups: AI and Human; one line per (layer, dir_kind); shaded ±1 std band.

probe_type_plot = "authorship"
metric_col      = "argmax_score"   # or "expected_score"
groups_plot     = ["AI", "Human"]

sub_smp = smp_df[smp_df["probe_type"] == probe_type_plot].copy()

layers_smp   = sorted(sub_smp["steer_layer"].unique())
dir_kinds_smp = sorted(sub_smp["dir_kind"].unique())
alphas_sorted = sorted(sub_smp["alpha"].unique())

# Color per layer (tab10), dash per dir_kind
_tab10 = [
    "rgb(31,119,180)", "rgb(255,127,14)", "rgb(44,160,44)",
    "rgb(214,39,40)",  "rgb(148,103,189)", "rgb(140,86,75)",
    "rgb(227,119,194)","rgb(127,127,127)", "rgb(188,189,34)", "rgb(23,190,207)",
]
_dash_style = {"elasticnet": "solid", "logistic": "dash"}
_group_symbol = {"AI": "circle", "Human": "square"}

# One subplot per group so lines don't overlap
from plotly.subplots import make_subplots as _msp

fig = _msp(
    rows=1, cols=len(groups_plot),
    subplot_titles=[f"{g} responses" for g in groups_plot],
    horizontal_spacing=0.08,
    shared_yaxes=True,
)

shown = set()
for li, sl in enumerate(layers_smp):
    color = _tab10[li % len(_tab10)]
    for dk in dir_kinds_smp:
        dash = _dash_style.get(dk, "solid")
        for ci, grp in enumerate(groups_plot):
            col = ci + 1
            means, stds = [], []
            for a in alphas_sorted:
                vals = sub_smp[
                    (sub_smp["steer_layer"] == sl) &
                    (sub_smp["dir_kind"]    == dk) &
                    (sub_smp["alpha"]       == a) &
                    (sub_smp["group"]       == grp)
                ][metric_col].dropna().values
                means.append(float(np.mean(vals)) if len(vals) else np.nan)
                stds.append(float(np.std(vals))  if len(vals) else np.nan)

            means = np.array(means)
            stds  = np.array(stds)
            label = f"L{sl} {dk}"
            show_leg = label not in shown and col == 1
            shown.add(label)

            # Shaded band ±1 std
            fig.add_trace(go.Scatter(
                x=alphas_sorted + alphas_sorted[::-1],
                y=np.concatenate([means + stds, (means - stds)[::-1]]).tolist(),
                fill="toself",
                fillcolor=color.replace("rgb", "rgba").replace(")", ",0.12)"),
                line=dict(color="rgba(0,0,0,0)"),
                showlegend=False, hoverinfo="skip",
            ), row=1, col=col)

            # Mean line
            fig.add_trace(go.Scatter(
                x=alphas_sorted, y=means.tolist(),
                mode="lines+markers",
                name=label,
                showlegend=show_leg,
                legendgroup=label,
                line=dict(color=color, dash=dash, width=2.5),
                marker=dict(size=6, symbol=_group_symbol.get(grp, "circle")),
            ), row=1, col=col)

# Reference: α=0 original Ollama mean per group
for ci, grp in enumerate(groups_plot):
    col = ci + 1
    orig_mean = orig_df[orig_df["group"] == grp]["accuracy_score"].mean()
    xref = "x" if col == 1 else f"x{col}"
    yref = "y" if col == 1 else f"y{col}"
    fig.add_shape(
        type="line",
        x0=alphas_sorted[0], x1=alphas_sorted[-1], y0=orig_mean, y1=orig_mean,
        line=dict(color="black", dash="dot", width=1.5),
        xref=xref, yref=yref,
    )
    fig.add_annotation(
        x=alphas_sorted[-1], y=orig_mean,
        text=f"Ollama mean ({orig_mean:.2f})",
        showarrow=False, xanchor="right",
        font=dict(size=13, family="Arial", color="black"),
        xref=xref, yref=yref,
    )

fig.update_layout(
    font=dict(size=18, family="Arial", color="black"),
    height=520, width=1100,
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(
        text=f"Mean ± std of {metric_col} per group — {probe_type_plot} probe<br>"
             "<span style='font-size:14px'>solid=elasticnet · dashed=logistic · dotted=Ollama baseline</span>",
        font=dict(size=20, family="Arial"),
    ),
    legend=dict(
        orientation="h", y=1.14, x=0.5, xanchor="center",
        font=dict(size=16, family="Arial"),
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="black", borderwidth=1.5,
    ),
    margin=dict(t=110, b=80, l=100, r=60),
)

for ci, grp in enumerate(groups_plot):
    col = ci + 1
    yax = "yaxis"  if col == 1 else f"yaxis{col}"
    xax = "xaxis"  if col == 1 else f"xaxis{col}"
    fig.layout[yax].update(
        title=dict(text=metric_col if col == 1 else "",
                   font=dict(size=20, family="Arial")),
        range=[3.5, 8.0],
        showgrid=False, zeroline=False,
        tickfont=dict(size=18, family="Arial"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
    )
    fig.layout[xax].update(
        title=dict(text="Alpha", font=dict(size=20, family="Arial")),
        showgrid=False, zeroline=False,
        tickfont=dict(size=18, family="Arial"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
    )

for ann in fig.layout.annotations[:len(groups_plot)]:
    ann.font.size = 20; ann.font.family = "Arial"

fig.show()

In [ ]:
# ── Mean ± std — ASYMMETRIC experiment ───────────────────────────────────────
probe_type_asym = "authorship"
metric_col_asym = "argmax_score"   # or "expected_score"
groups_asym     = ["AI", "Human"]

sub_asym = asym_smp_df[asym_smp_df["probe_type"] == probe_type_asym].copy()
layers_asym    = sorted(sub_asym["steer_layer"].unique())
dir_kinds_asym = sorted(sub_asym["dir_kind"].unique())
alphas_asym    = sorted(sub_asym["alpha"].unique())

_tab10 = [
    "rgb(31,119,180)", "rgb(255,127,14)", "rgb(44,160,44)",
    "rgb(214,39,40)",  "rgb(148,103,189)", "rgb(140,86,75)",
    "rgb(227,119,194)","rgb(127,127,127)", "rgb(188,189,34)", "rgb(23,190,207)",
]
_dash_style_a = {"elasticnet": "solid", "logistic": "dash"}

from plotly.subplots import make_subplots as _msp2

fig = _msp2(
    rows=1, cols=len(groups_asym),
    subplot_titles=[f"{g} responses" for g in groups_asym],
    horizontal_spacing=0.08,
    shared_yaxes=True,
)

shown_a = set()
for li, sl in enumerate(layers_asym):
    color = _tab10[li % len(_tab10)]
    for dk in dir_kinds_asym:
        dash = _dash_style_a.get(dk, "solid")
        for ci, grp in enumerate(groups_asym):
            col = ci + 1
            means, stds = [], []
            for a in alphas_asym:
                vals = sub_asym[
                    (sub_asym["steer_layer"] == sl) &
                    (sub_asym["dir_kind"]    == dk) &
                    (sub_asym["alpha"]       == a) &
                    (sub_asym["group"]       == grp)
                ][metric_col_asym].dropna().values
                means.append(float(np.mean(vals)) if len(vals) else np.nan)
                stds.append(float(np.std(vals))  if len(vals) else np.nan)

            means = np.array(means); stds = np.array(stds)
            label = f"L{sl} {dk}"
            show_leg = label not in shown_a and col == 1
            shown_a.add(label)

            # Shaded ±1 std
            fig.add_trace(go.Scatter(
                x=alphas_asym + alphas_asym[::-1],
                y=np.concatenate([means + stds, (means - stds)[::-1]]).tolist(),
                fill="toself",
                fillcolor=color.replace("rgb", "rgba").replace(")", ",0.12)"),
                line=dict(color="rgba(0,0,0,0)"),
                showlegend=False, hoverinfo="skip",
            ), row=1, col=col)

            fig.add_trace(go.Scatter(
                x=alphas_asym, y=means.tolist(),
                mode="lines+markers",
                name=label,
                showlegend=show_leg,
                legendgroup=label,
                line=dict(color=color, dash=dash, width=2.5),
                marker=dict(size=6),
            ), row=1, col=col)

# Ollama reference line (AI = AI + CoT AI)
for ci, grp in enumerate(groups_asym):
    col = ci + 1
    orig_mean = float(np.mean(get_orig_scores(grp)))
    xref = "x" if col == 1 else f"x{col}"
    yref = "y" if col == 1 else f"y{col}"
    fig.add_shape(
        type="line",
        x0=alphas_asym[0], x1=alphas_asym[-1], y0=orig_mean, y1=orig_mean,
        line=dict(color="black", dash="dot", width=1.5),
        xref=xref, yref=yref,
    )
    fig.add_annotation(
        x=alphas_asym[-1], y=orig_mean,
        text=f"Ollama mean ({orig_mean:.2f})",
        showarrow=False, xanchor="right",
        font=dict(size=13, family="Arial", color="black"),
        xref=xref, yref=yref,
    )

fig.update_layout(
    font=dict(size=18, family="Arial", color="black"),
    height=520, width=1100,
    template="plotly_white",
    plot_bgcolor="white", paper_bgcolor="white",
    title=dict(
        text=(f"Asymmetric steering — Mean ± std of {metric_col_asym} — {probe_type_asym}<br>"
              "<span style='font-size:14px'>AI: h−α·v · Human: h+α·v · "
              "solid=elasticnet · dashed=logistic · dotted=Ollama</span>"),
        font=dict(size=20, family="Arial"),
    ),
    legend=dict(
        orientation="h", y=1.14, x=0.5, xanchor="center",
        font=dict(size=16, family="Arial"),
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="black", borderwidth=1.5,
    ),
    margin=dict(t=120, b=80, l=100, r=60),
)

for ci, grp in enumerate(groups_asym):
    col = ci + 1
    yax = "yaxis"  if col == 1 else f"yaxis{col}"
    xax = "xaxis"  if col == 1 else f"xaxis{col}"
    fig.layout[yax].update(
        title=dict(text=metric_col_asym if col == 1 else "",
                   font=dict(size=20, family="Arial")),
        range=[3.5, 8.0],
        showgrid=False, zeroline=False,
        tickfont=dict(size=18, family="Arial"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
    )
    fig.layout[xax].update(
        title=dict(text="Alpha", font=dict(size=20, family="Arial")),
        showgrid=False, zeroline=False,
        tickfont=dict(size=18, family="Arial"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
    )

for ann in fig.layout.annotations[:len(groups_asym)]:
    ann.font.size = 20; ann.font.family = "Arial"

fig.show()

## 6. Quality vs alpha: Spearman ρ (AI and Human)

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Quality — AI group", "Quality — Human group"],
                    shared_yaxes=True)

sub = agg_df[agg_df["probe_type"] == "authorship"]

for li, sl in enumerate(layers):
    for dk in dir_kinds:
        sub2 = sub[(sub["steer_layer"] == sl) & (sub["dir_kind"] == dk)].sort_values("alpha")
        if sub2.empty:
            continue
        c = _colors_tab10[li % len(_colors_tab10)]
        d = _dash.get(dk, "solid")
        name = f"L{sl} {dk}"
        fig.add_trace(go.Scatter(
            x=sub2["alpha"], y=sub2["quality_corr_ai"],
            mode="lines+markers", name=name, showlegend=True,
            line=dict(color=c, dash=d, width=2.5), marker=dict(size=6),
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=sub2["alpha"], y=sub2["quality_corr_human"],
            mode="lines+markers", name=name, showlegend=False,
            line=dict(color=c, dash=d, width=2.5), marker=dict(size=6),
        ), row=1, col=2)

for col in [1, 2]:
    yax = "yaxis" if col == 1 else "yaxis2"
    xax = "xaxis" if col == 1 else "xaxis2"
    fig.layout[yax].update(title="Spearman ρ" if col == 1 else "",
                            showgrid=False, range=[0, 1],
                            showline=True, linewidth=2, linecolor="black")
    fig.layout[xax].update(title="Alpha",
                            showgrid=False,
                            showline=True, linewidth=2, linecolor="black")

fig.update_layout(
    title="Quality preservation (Spearman ρ) vs alpha",
    font=dict(size=18, family="Arial"),
    template="plotly_white",
    height=500, width=1200,
    legend=dict(orientation="h", y=1.15, x=0.5, xanchor="center"),
)
fig.show()

## 7. Export SVG

In [ ]:
# Run once to generate the main figure for the paper
EXPORT_DIR = ROOT / "results" / "elastic_plots"

for metric in ["argmax_score", "expected_score"]:
    for sl in [15]:  # change to desired layer
        fig = steering_violin_plot(
            probe_type="authorship", dir_kind="elasticnet",
            steer_layer=sl, metric=metric,
            groups=["Human", "AI"],
            show_original=True,
            height=700, width=1600,
        )
        out = EXPORT_DIR / f"{MODEL_KEY}_steering_violin_L{sl}_{metric}.svg"
        fig.write_image(str(out), scale=3)
        print(f"Saved: {out.name}")

---

## 8. Asymmetric Steering Experiment (L15)

**AI side**: `h' = h − α·v` (push away from AI direction)  
**Human side**: `h' = h + α·v` (push toward AI direction)

Both directions compared: logistic vs elastic-net probe at L15.

In [ ]:
# ── Load asymmetric steering CSVs ────────────────────────────────────────────
asym_smp_path = STEER_DIR / f"{MODEL_KEY}_asym_steering_samples.csv"
asym_agg_path = STEER_DIR / f"{MODEL_KEY}_asym_steering.csv"

if not asym_smp_path.exists():
    raise FileNotFoundError(
        f"Asymmetric steering results not found: {asym_smp_path}\n"
        "Run: python scripts/run_gpu_steps.py --from_step 19"
    )

asym_smp_df = pd.read_csv(asym_smp_path)
asym_agg_df = pd.read_csv(asym_agg_path)

print(f"Asymmetric samples: {len(asym_smp_df)} rows")
print(f"  probe_types:  {asym_smp_df['probe_type'].unique().tolist()}")
print(f"  dir_kinds:    {asym_smp_df['dir_kind'].unique().tolist()}")
print(f"  steer_layers: {sorted(asym_smp_df['steer_layer'].unique().tolist())}")
print(f"  alphas:       {sorted(asym_smp_df['alpha'].unique().tolist())}")
print(f"  groups:       {asym_smp_df['group'].unique().tolist()}")

def best_alpha_asym(probe_type, dir_kind, steer_layer):
    """Alpha that maximally reduces |bias| in the asymmetric experiment."""
    sub = asym_agg_df[
        (asym_agg_df["probe_type"]  == probe_type) &
        (asym_agg_df["dir_kind"]    == dir_kind) &
        (asym_agg_df["steer_layer"] == steer_layer)
    ].copy()
    if sub.empty:
        return None
    base = sub.loc[sub["alpha"] == 0.0, "bias_delta"].values
    baseline = float(base[0]) if len(base) else float(sub["bias_delta"].iloc[0])
    sub["bias_red"] = (baseline - sub["bias_delta"]).abs()
    return float(sub.loc[sub["bias_red"].idxmax(), "alpha"])

In [ ]:
def asym_violin_plot(
    probe_type    = "authorship",
    dir_kind      = "elasticnet",
    steer_layer   = 15,
    metric        = "argmax_score",
    groups        = ["Human", "AI"],
    selected_alphas = None,
    show_original = True,
    height = 900,
    width  = None,
):
    """
    Violin plot for the asymmetric steering experiment.
    AI responses were steered with h - α·v, Human with h + α·v.
    Same Nature style as steering_violin_plot().
    """
    if selected_alphas is None:
        all_alphas = sorted(asym_smp_df["alpha"].unique())
        ba = best_alpha_asym(probe_type, dir_kind, steer_layer) or 0.0
        ma = max(all_alphas)
        selected_alphas = sorted(set([0.0, ba, ma]))

    conditions = []
    if show_original:
        conditions.append({"label": "Original\n(Ollama)", "source": "original", "alpha": None})
    for α in selected_alphas:
        label = f"α={α:.1f}" + (" (baseline)" if α == 0 else "")
        conditions.append({"label": label, "source": "asym", "alpha": α})

    n_cols = len(conditions)
    if width is None:
        width = max(900, n_cols * len(groups) * 280)

    metric_title = {"argmax_score":   "Score (argmax, 1–7)",
                    "expected_score": "Expected score Σ d·P(d)"}.get(metric, metric)

    fig = make_subplots(
        rows=1, cols=n_cols,
        subplot_titles=[c["label"] for c in conditions],
        horizontal_spacing=0.04,
        shared_yaxes=True,
    )

    # ── Data collection ────────────────────────────────────────────────────
    group_data_by_cond = {}
    for ci, cond in enumerate(conditions):
        gd = {}
        for grp in groups:
            if cond["source"] == "original":
                vals = orig_df[orig_df["group"] == grp]["accuracy_score"].dropna().values
            else:
                sub = asym_smp_df[
                    (asym_smp_df["probe_type"]  == probe_type) &
                    (asym_smp_df["dir_kind"]    == dir_kind) &
                    (asym_smp_df["steer_layer"] == steer_layer) &
                    (asym_smp_df["alpha"]       == cond["alpha"]) &
                    (asym_smp_df["group"]       == grp)
                ]
                vals = sub[metric].dropna().values if not sub.empty else np.array([])
            gd[grp] = vals
        group_data_by_cond[ci] = gd

    # ── Global FDR correction ──────────────────────────────────────────────
    pv_infos = []
    for ci in range(n_cols):
        gd = group_data_by_cond[ci]
        for g1, g2 in [("Human", "AI"), ("Human", "CoT AI"), ("AI", "CoT AI")]:
            if len(gd.get(g1, [])) < 2 or len(gd.get(g2, [])) < 2:
                continue
            p = mw_pvalue(gd[g1], gd[g2])
            if not np.isnan(p):
                pv_infos.append({"ci": ci, "g1": g1, "g2": g2, "p_raw": p})

    if pv_infos:
        _, pv_adj, _, _ = multipletests(
            [x["p_raw"] for x in pv_infos], alpha=0.05, method="fdr_bh")
        for i, info in enumerate(pv_infos):
            info["p_adj"] = pv_adj[i]

    pv_lookup = {(x["ci"], x["g1"], x["g2"]): x.get("p_adj", x["p_raw"])
                 for x in pv_infos}

    # ── Draw violins ──────────────────────────────────────────────────────
    x_pos        = {g: i for i, g in enumerate(groups)}
    shown_legend = set()

    for ci, cond in enumerate(conditions):
        col = ci + 1
        gd  = group_data_by_cond[ci]

        for grp in groups:
            vals = gd.get(grp, np.array([]))
            if len(vals) == 0:
                continue
            color    = GROUP_COLORS.get(grp, "gray")
            show_leg = grp not in shown_legend
            shown_legend.add(grp)

            fig.add_trace(
                go.Violin(
                    y=vals,
                    name=f"{grp} (n={len(vals)})",
                    x0=x_pos[grp],
                    fillcolor=color,
                    opacity=0.7,
                    box_visible=True,
                    meanline_visible=True,
                    width=0.5,
                    showlegend=show_leg,
                    legendgroup=grp,
                    line=dict(width=2, color="black"),
                    marker=dict(color=color, line=dict(color="black", width=1)),
                    scalemode="width",
                ),
                row=1, col=col,
            )

        # ── Brackets ──────────────────────────────────────────────────────
        xref, yref = subplot_refs(ci + 1, 1, n_cols)
        for (g1, g2), y_h in BRACKET_HEIGHTS.items():
            if g1 not in x_pos or g2 not in x_pos:
                continue
            p = pv_lookup.get((ci, g1, g2)) or pv_lookup.get((ci, g2, g1))
            if p is None:
                continue
            add_bracket(fig, x_pos[g1], x_pos[g2], y_h, fmt_p(p), xref, yref)

    # ── Layout ────────────────────────────────────────────────────────────
    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white",
        plot_bgcolor="white",
        paper_bgcolor="white",
        title=dict(
            text=(f"Asymmetric steering — {probe_type} | {dir_kind} | L{steer_layer} | {metric}<br>"
                  "<span style='font-size:18px'>AI: h−α·v  |  Human: h+α·v</span>"),
            font=dict(size=22, family="Arial"),
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08,
            xanchor="center", x=0.5,
            font=dict(size=22, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=110, b=80, l=100, r=60),
    )

    for ci in range(n_cols):
        col = ci + 1
        yax = "yaxis"  if col == 1 else f"yaxis{col}"
        xax = "xaxis"  if col == 1 else f"xaxis{col}"
        fig.layout[yax].update(
            title=dict(text=metric_title if col == 1 else "",
                       font=dict(size=26, family="Arial", color="black")),
            range=[0, 9.5],
            showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        )
        fig.layout[xax].update(
            ticktext=groups, tickvals=list(range(len(groups))),
            showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        )

    for ann in fig.layout.annotations[:n_cols]:
        ann.font.size   = 26
        ann.font.family = "Arial"
        ann.font.color  = "black"

    fig.show()
    return fig

print("asym_violin_plot() defined.")

### 8.1  Authorship — Elastic-net direction, L15, argmax

In [ ]:
fig = asym_violin_plot(probe_type="authorship", dir_kind="elasticnet", steer_layer=15, metric="argmax_score")

### 8.2  Authorship — Logistic direction, L15, argmax

In [ ]:
fig = asym_violin_plot(probe_type="authorship", dir_kind="logistic", steer_layer=15, metric="argmax_score")

### 8.3  Expected score — Elastic-net, L15

In [ ]:
fig = asym_violin_plot(probe_type="authorship", dir_kind="elasticnet", steer_layer=15, metric="expected_score")

### 8.4  Verbosity probe — Asymmetric, L15

In [ ]:
for dk in ["elasticnet", "logistic"]:
    for metric in ["argmax_score", "expected_score"]:
        fig = asym_violin_plot(probe_type="verbosity", dir_kind=dk, steer_layer=15, metric=metric)
        fig.update_layout(title_text=f"Asymmetric — verbosity | {dk} | L15 | {metric}")

### 8.5  Tertile analysis — Verbosity probe, Asymmetric L15, best alpha

At the alpha that maximally reduces bias, do longer responses still receive inflated scores?  
Responses split into tertiles by word count: **T1 (short) · T2 (medium) · T3 (long)**.

In [ ]:
# ── Load dataset and compute response lengths ─────────────────────────────────
from src.data import load_cardio_dataset, get_unique_responses

raw_ds = load_cardio_dataset(cfg)
ds = get_unique_responses(
    raw_ds,
    target_evaluator=cfg.dataset.get("target_evaluator", "Llama3.1:8b"),
    filter_uninformative=True,
)
n_ds = len(ds)

ai_sources_set    = set(cfg.dataset["ai_sources"])    # {"AI", "CoT AI"}
human_sources_set = set(cfg.dataset["human_sources"]) # {"Human"}

# Responses in dataset order, split by group (same order as steering experiment)
ai_resps    = [(i, ds[i]["response"]) for i in range(n_ds) if ds[i]["response_source"] in ai_sources_set]
human_resps = [(i, ds[i]["response"]) for i in range(n_ds) if ds[i]["response_source"] in human_sources_set]

ai_lengths    = np.array([len(r.split()) for _, r in ai_resps])
human_lengths = np.array([len(r.split()) for _, r in human_resps])

# Global tertile boundaries over ALL responses (AI + Human combined)
all_lengths = np.concatenate([ai_lengths, human_lengths])
q33, q66    = np.percentile(all_lengths, [33.33, 66.67])

def assign_tertile(lengths):
    t = np.zeros(len(lengths), dtype=int)
    t[lengths > q33] = 1
    t[lengths > q66] = 2
    return t

ai_tertiles    = assign_tertile(ai_lengths)
human_tertiles = assign_tertile(human_lengths)

print(f"Dataset: {n_ds} unique responses  (AI+CoT={len(ai_resps)}, Human={len(human_resps)})")
print(f"Tertile boundaries (words): T1 ≤{q33:.0f}  |  T2 {q33:.0f}–{q66:.0f}  |  T3 >{q66:.0f}")
print(f"AI     tertile counts: {np.bincount(ai_tertiles)}")
print(f"Human  tertile counts: {np.bincount(human_tertiles)}")

In [ ]:
TERT_PROBE  = "verbosity"
TERT_LAYER  = 15
TERT_DIR    = "elasticnet"
TERT_BEST_A = best_alpha_asym(TERT_PROBE, TERT_DIR, TERT_LAYER)
print(f"Best alpha (verbosity, {TERT_DIR}, L{TERT_LAYER}): {TERT_BEST_A}")

TERTILE_LABELS = ["T1 (Short)", "T2 (Medium)", "T3 (Long)"]
TERTILE_COLORS = ["#2166ac", "#4dac26", "#d01c8b"]

def build_tertile_df(probe_type, dir_kind, steer_layer, alpha, df_src):
    """
    Join steered scores with tertile labels by positional match within each group.
    Returns DataFrame: group, argmax_score, expected_score, tertile, tertile_label
    """
    rows = []
    for grp, tertiles_arr in [("AI", ai_tertiles), ("Human", human_tertiles)]:
        sub = df_src[
            (df_src["probe_type"]  == probe_type) &
            (df_src["dir_kind"]    == dir_kind) &
            (df_src["steer_layer"] == steer_layer) &
            (df_src["alpha"]       == alpha) &
            (df_src["group"]       == grp)
        ].reset_index(drop=True)
        if sub.empty:
            continue
        n_match = min(len(sub), len(tertiles_arr))
        if len(sub) != len(tertiles_arr):
            print(f"  [WARN] {grp} α={alpha}: steering {len(sub)} vs dataset {len(tertiles_arr)} → using {n_match}")
        for idx in range(n_match):
            rows.append({
                "group":          grp,
                "argmax_score":   sub.loc[idx, "argmax_score"],
                "expected_score": sub.loc[idx, "expected_score"],
                "tertile":        int(tertiles_arr[idx]),
                "tertile_label":  TERTILE_LABELS[int(tertiles_arr[idx])],
            })
    return pd.DataFrame(rows)


# Elasticnet direction
tert_df_a0   = build_tertile_df(TERT_PROBE, TERT_DIR, TERT_LAYER, 0.0,         asym_smp_df)
tert_df_best = build_tertile_df(TERT_PROBE, TERT_DIR, TERT_LAYER, TERT_BEST_A, asym_smp_df)
print(f"Elasticnet — α=0: {len(tert_df_a0)} rows | α={TERT_BEST_A}: {len(tert_df_best)} rows")

# Logistic direction
TERT_DIR_LOG    = "logistic"
TERT_BEST_A_LOG = best_alpha_asym(TERT_PROBE, TERT_DIR_LOG, TERT_LAYER)
print(f"Best alpha (verbosity, {TERT_DIR_LOG}, L{TERT_LAYER}): {TERT_BEST_A_LOG}")

tert_df_a0_log   = build_tertile_df(TERT_PROBE, TERT_DIR_LOG, TERT_LAYER, 0.0,            asym_smp_df)
tert_df_best_log = build_tertile_df(TERT_PROBE, TERT_DIR_LOG, TERT_LAYER, TERT_BEST_A_LOG, asym_smp_df)
print(f"Logistic    — α=0: {len(tert_df_a0_log)} rows | α={TERT_BEST_A_LOG}: {len(tert_df_best_log)} rows")

print("\nMean scores (elasticnet, α=best):")
print(tert_df_best.groupby(["group","tertile_label"])[["argmax_score","expected_score"]].mean().round(3))

In [ ]:
def tertile_violin_plot(
    tert_df_a0,
    tert_df_best,
    alpha_best,
    metric      = "argmax_score",
    probe_type  = TERT_PROBE,
    dir_kind    = TERT_DIR,
    steer_layer = TERT_LAYER,
    height = 500,
    width  = 800,
):
    """
    Score distributions by response-length tertile (T1/T2/T3).
    Within each tertile: α=0 (light) and α=best (solid) violin side by side.
    Two subplots: AI group | Human group.
    N shown above each violin. P-values on best-α tertile pairs (FDR-BH).
    """
    groups = ["AI", "Human"]
    metric_label = {"argmax_score": "Score (argmax, 1–7)",
                    "expected_score": "Expected score Σ d·P(d)"}.get(metric, metric)

    # X positions: T0=[0, 0.6], T1=[2.2, 2.8], T2=[4.4, 5.0]
    def xpos(t_idx, is_best):
        return t_idx * 2.2 + (0.6 if is_best else 0.0)

    tick_vals  = [xpos(t, False) + 0.3 for t in range(3)]
    tick_texts = TERTILE_LABELS

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[f"{g} responses" for g in groups],
                        horizontal_spacing=0.08, shared_yaxes=True)

    # Collect data
    cell_data = {}
    for grp in groups:
        for t_idx in range(3):
            for is_best, df_src in [(False, tert_df_a0), (True, tert_df_best)]:
                vals = df_src[
                    (df_src["group"] == grp) & (df_src["tertile"] == t_idx)
                ][metric].dropna().values
                cell_data[(grp, t_idx, is_best)] = vals

    # P-values: best-α tertile comparisons T0 vs T1, T1 vs T2, T0 vs T2
    pv_infos = []
    for ci, grp in enumerate(groups):
        for t1, t2 in [(0, 1), (1, 2), (0, 2)]:
            d1 = cell_data.get((grp, t1, True), np.array([]))
            d2 = cell_data.get((grp, t2, True), np.array([]))
            p = mw_pvalue(d1, d2)
            if not np.isnan(p):
                pv_infos.append({"grp": grp, "ci": ci, "t1": t1, "t2": t2, "p_raw": p})

    if pv_infos:
        _, pv_adj, _, _ = multipletests(
            [x["p_raw"] for x in pv_infos], alpha=0.05, method="fdr_bh")
        for i, info in enumerate(pv_infos):
            info["p_adj"] = pv_adj[i]
    pv_lookup = {(x["ci"], x["t1"], x["t2"]): x.get("p_adj", x["p_raw"])
                 for x in pv_infos}

    # Bracket heights (below n annotation at y≈9.0)
    bracket_heights = {(0, 1): 6.8, (1, 2): 7.5, (0, 2): 8.2}

    shown = set()
    for ci, grp in enumerate(groups):
        col = ci + 1
        xref = "x" if col == 1 else f"x{col}"
        yref = "y" if col == 1 else f"y{col}"

        for t_idx in range(3):
            color = TERTILE_COLORS[t_idx]
            for is_best in [False, True]:
                vals    = cell_data.get((grp, t_idx, is_best), np.array([]))
                opacity = 0.7 if is_best else 0.3
                alpha_lbl  = f"α={alpha_best:.1f}" if is_best else "α=0"
                legend_key = f"{TERTILE_LABELS[t_idx]} {alpha_lbl}"
                show_leg = legend_key not in shown
                shown.add(legend_key)
                xv = xpos(t_idx, is_best)

                fig.add_trace(go.Violin(
                    y=vals,
                    name=legend_key,
                    x0=xv,
                    fillcolor=color,
                    opacity=opacity,
                    box_visible=True,
                    meanline_visible=True,
                    width=0.5,
                    showlegend=show_leg,
                    legendgroup=legend_key,
                    line=dict(width=2, color="black"),
                    marker=dict(color=color, line=dict(color="black", width=1)),
                    scalemode="width",
                ), row=1, col=col)

                # n annotation above violin
                fig.add_annotation(
                    x=xv, y=9.05, text=f"n={len(vals)}",
                    showarrow=False,
                    font=dict(size=11, family="Arial", color="black"),
                    xref=xref, yref=yref,
                )

        # Brackets between best-α tertile pairs
        for (t1, t2), y_h in bracket_heights.items():
            p = pv_lookup.get((ci, t1, t2))
            if p is None:
                continue
            add_bracket(fig, xpos(t1, True), xpos(t2, True), y_h, fmt_p(p), xref, yref)

    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
        title=dict(
            text=(f"Tertile analysis — {probe_type} | {dir_kind} | L{steer_layer} | "
                  f"α=0 (light) vs α={alpha_best:.1f} (solid) | {metric}"),
            font=dict(size=20, family="Arial"),
            y=0.98,
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
            font=dict(size=16, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=100, b=80, l=100, r=60),
    )

    for ci, grp in enumerate(groups):
        col = ci + 1
        yax = "yaxis"  if col == 1 else f"yaxis{col}"
        xax = "xaxis"  if col == 1 else f"xaxis{col}"
        fig.layout[yax].update(
            title=dict(text=metric_label if col == 1 else "",
                       font=dict(size=26, family="Arial", color="black")),
            range=[0, 9.5], showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        )
        fig.layout[xax].update(
            ticktext=tick_texts, tickvals=tick_vals,
            showgrid=False, zeroline=False,
            tickfont=dict(size=18, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
            range=[-0.4, 5.4],
        )
    for ann in fig.layout.annotations[:2]:
        ann.font.size = 26; ann.font.family = "Arial"; ann.font.color = "black"
    
    fig.show()
    import importlib


    fig.write_image(str(f"{MODEL_KEY}_steering_tertile_violin_{dir_kind}_L{steer_layer}_{metric}.svg"), scale=3)
    return fig


# ── Run for both directions and both metrics ──────────────────────────────────
for dk, df_a0, df_best, best_a in [
    ("elasticnet", tert_df_a0,     tert_df_best,     TERT_BEST_A),
    ("logistic",   tert_df_a0_log, tert_df_best_log, TERT_BEST_A_LOG),
]:
    for metric in ["argmax_score", "expected_score"]:
        tertile_violin_plot(df_a0, df_best, best_a, metric=metric, dir_kind=dk)

In [ ]:
def tertile_group_violin_plot(
    tert_df_a0,
    tert_df_best,
    alpha_best,
    metric      = "argmax_score",
    probe_type  = TERT_PROBE,
    dir_kind    = TERT_DIR,
    steer_layer = TERT_LAYER,
    height = 500,
    width  = 800,
):
    """
    For each tertile: Human_α0, Human_αbest, AI_α0, AI_αbest side by side.
    Single plot with vertical separators between tertile blocks.
    N shown above each violin.
    P-values: Human vs AI for α=0 (lower bracket) and α=best (upper bracket), FDR-BH.
    """
    metric_label = {"argmax_score": "Score (argmax, 1–7)",
                    "expected_score": "Expected score Σ d·P(d)"}.get(metric, metric)

    # Within each tertile block: H_α0=0, H_αbest=0.65, [gap 0.7], A_α0=1.35, A_αbest=2.0
    # Block stride = 3.7 (last pos 2.0, next block starts at 3.7)
    BLOCK = 3.7

    def xpos(t_idx, grp, is_best):
        within = {"Human": {False: 0.0, True: 0.65},
                  "AI":    {False: 1.35, True: 2.0}}
        return t_idx * BLOCK + within[grp][is_best]

    group_colors = {"Human": GROUP_COLORS["Human"], "AI": GROUP_COLORS["AI"]}

    fig = go.Figure()

    # Collect data
    cell_data = {}
    for grp in ["Human", "AI"]:
        for t_idx in range(3):
            for is_best, df_src in [(False, tert_df_a0), (True, tert_df_best)]:
                vals = df_src[
                    (df_src["group"] == grp) & (df_src["tertile"] == t_idx)
                ][metric].dropna().values
                cell_data[(grp, t_idx, is_best)] = vals

    # P-values: Human vs AI within each tertile, for both α=0 and α=best
    pv_infos = []
    for t_idx in range(3):
        for is_best in [False, True]:
            dH = cell_data.get(("Human", t_idx, is_best), np.array([]))
            dA = cell_data.get(("AI",    t_idx, is_best), np.array([]))
            p  = mw_pvalue(dH, dA)
            if not np.isnan(p):
                pv_infos.append({
                    "t_idx":   t_idx,
                    "is_best": is_best,
                    "p_raw":   p,
                    "xH":      xpos(t_idx, "Human", is_best),
                    "xA":      xpos(t_idx, "AI",    is_best),
                })

    if pv_infos:
        _, pv_adj, _, _ = multipletests(
            [x["p_raw"] for x in pv_infos], alpha=0.05, method="fdr_bh")
        for i, info in enumerate(pv_infos):
            info["p_adj"] = pv_adj[i]

    # Draw violins
    shown = set()
    for grp in ["Human", "AI"]:
        for t_idx in range(3):
            for is_best in [False, True]:
                vals    = cell_data.get((grp, t_idx, is_best), np.array([]))
                color   = group_colors[grp]
                opacity = 0.7 if is_best else 0.3
                alpha_lbl  = f"α={alpha_best:.1f}" if is_best else "α=0"
                legend_key = f"{grp} {alpha_lbl}"
                show_leg   = legend_key not in shown
                shown.add(legend_key)
                xv = xpos(t_idx, grp, is_best)

                fig.add_trace(go.Violin(
                    y=vals,
                    name=legend_key,
                    x0=xv,
                    fillcolor=color,
                    opacity=opacity,
                    box_visible=True,
                    meanline_visible=True,
                    width=0.5,
                    showlegend=show_leg,
                    legendgroup=legend_key,
                    line=dict(width=2, color="black"),
                    marker=dict(color=color, line=dict(color="black", width=1)),
                    scalemode="width",
                ))

                """# n annotation above violin
                fig.add_annotation(
                    x=xv, y=9.15, text=f"n={len(vals)}",
                    showarrow=False,
                    font=dict(size=11, family="Arial", color="black"),
                    xref="x", yref="y",
                )
                """

    # P-value brackets: α=0 lower (y=7.3), α=best higher (y=8.1)
    for info in pv_infos:
        y_h = 8.1 if info["is_best"] else 7.3
        add_bracket(fig, info["xH"], info["xA"], y_h, fmt_p(info["p_adj"]), "x", "y")

    # Vertical separators between tertile blocks
    for t_idx in range(1, 3):
        x_sep = t_idx * BLOCK - 0.85
        fig.add_shape(type="line", x0=x_sep, y0=0, x1=x_sep, y1=9.5,
                      line=dict(color="gray", width=1, dash="dot"),
                      xref="x", yref="y")

    # Tertile labels at top of each block
    for t_idx in range(3):
        xc = (xpos(t_idx, "Human", False) + xpos(t_idx, "AI", True)) / 2
        fig.add_annotation(
            x=xc, y=9.65, text=TERTILE_LABELS[t_idx],
            showarrow=False,
            font=dict(size=17, family="Arial", color=TERTILE_COLORS[t_idx]),
            xref="x", yref="y",
        )

    x_max = xpos(2, "AI", True) + 0.55
    tick_vals  = [(xpos(t, "Human", False) + xpos(t, "AI", True)) / 2 for t in range(3)]
    tick_texts = [TERTILE_LABELS[t] for t in range(3)]

    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
        title=dict(
            text=(f"Tertile × group — {probe_type} | {dir_kind} | L{steer_layer} | "
                  f"α=0 (light) vs α={alpha_best:.1f} (solid) | {metric}"),
            font=dict(size=20, family="Arial"),
            y=0.98
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
            font=dict(size=18, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=100, b=80, l=100, r=60),
        xaxis=dict(
            ticktext=tick_texts, tickvals=tick_vals,
            showgrid=False, zeroline=False,
            tickfont=dict(size=20, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
            range=[-0.55, x_max],
        ),
        yaxis=dict(
            title=dict(text=metric_label, font=dict(size=16, family="Arial", color="black")),
            range=[0, 10.2], showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        ),
    )

    fig.show()
    fig.write_image(f"tertile_group_violin_{dir_kind}_L{steer_layer}_{metric}.svg", scale=3)
    return fig


# ── Run for both directions and both metrics ──────────────────────────────────
for dk, df_a0, df_best, best_a in [
    ("elasticnet", tert_df_a0,     tert_df_best,     TERT_BEST_A),
    ("logistic",   tert_df_a0_log, tert_df_best_log, TERT_BEST_A_LOG),
]:
    for metric in ["argmax_score", "expected_score"]:
        tertile_group_violin_plot(df_a0, df_best, best_a, metric=metric, dir_kind=dk)

---

## 8. Shuffle Experiment: Style Bias vs Content Bias

Each question is paired with a **random (mismatched) response** from the same group (AI or Human).  
If AI responses still score higher than Human responses even when paired with arbitrary questions,  
the bias is driven by **surface writing style**, not response quality.

Comparison: original Ollama scores vs. shuffled-pair model scores.

In [ ]:
# ── Load shuffle experiment results ─────────────────────────────────────────
SHUFFLE_JSON = TABLE_DIR / f"{MODEL_KEY}_shuffled_scores.json"

if not SHUFFLE_JSON.exists():
    raise FileNotFoundError(
        f"Shuffle results not found: {SHUFFLE_JSON}\n"
        "Run: python blocks/block8_probes/03_geometry_all_probes.py"
    )

with open(SHUFFLE_JSON) as f:
    shuffle_data = json.load(f)

shuffle_records = shuffle_data["records"]
shuffle_bias    = shuffle_data["bias_stats"]

# Build DataFrames
shuf_df = pd.DataFrame(shuffle_records)
print(f"Shuffle records: {len(shuf_df)}")
print(shuf_df.groupby("group")[["expected_score", "most_likely"]].describe().round(3))
print()
print("Bias stats (AI − Human on shuffled pairs):")
ai_minus_human = shuffle_bias.get("ai_minus_human", "N/A")
print(f"  AI − Human: {ai_minus_human}")
for grp in ["AI", "Human", "CoT AI"]:
    if grp in shuffle_bias:
        s = shuffle_bias[grp]
        print(f"  {grp:8s}: mean={s['mean']:.3f} ± {s['std']:.3f}  n={s['n']}")

In [ ]:
def shuffle_violin_plot(
    metric        = "most_likely",
    groups        = ["Human", "AI"],
    probe_type    = "authorship",
    dir_kind      = "elasticnet",
    steer_layer   = 15,
    height = 400,
    width  = 600,
):
    """
    Three-panel: Ollama original | Llama fwd (matched, α=0) | Llama fwd (shuffled).
    'AI' in original panel = AI + CoT AI combined.
    """
    STEER_COL = {"most_likely": "argmax_score", "expected_score": "expected_score"}
    steer_col = STEER_COL.get(metric, metric)

    metric_label = {"most_likely": "Score (argmax, 1–7)",
                    "expected_score": "Expected score Σ d·P(d)"}.get(metric, metric)

    conditions = [
        {"label": "Original (matched pairs)",       "source": "original"},
        {"label": "matched pairs, α=0",   "source": "steer_a0"},
        {"label": "shuffled pairs",        "source": "shuffle"},
    ]
    if width is None:
        width = max(1100, 3 * len(groups) * 280)

    steer_a0 = smp_df[
        (smp_df["probe_type"]  == probe_type) &
        (smp_df["dir_kind"]    == dir_kind) &
        (smp_df["steer_layer"] == steer_layer) &
        (smp_df["alpha"]       == 0.0)
    ] if not smp_df.empty else pd.DataFrame()

    group_data_by_cond = {}
    for ci, cond in enumerate(conditions):
        gd = {}
        for grp in groups:
            if cond["source"] == "original":
                vals = get_orig_scores(grp)
            elif cond["source"] == "steer_a0":
                sub  = steer_a0[steer_a0["group"] == grp]
                vals = sub[steer_col].dropna().values if not sub.empty else np.array([])
            else:
                sub  = shuf_df[shuf_df["group"] == grp]
                vals = sub[metric].dropna().values if not sub.empty else np.array([])
            gd[grp] = vals
        group_data_by_cond[ci] = gd

    title = f"Shuffle experiment — {MODEL_KEY} | {metric}"
    fig = _violin_figure(conditions, group_data_by_cond, groups, title, metric_label, height, width)
    fig.show()
    fig.write_image(f"{MODEL_KEY}_shuffle_violin_{metric}.svg", scale=3)
    return fig

print("shuffle_violin_plot() defined.")

### 8.1  Argmax score: Original (matched) vs Shuffled

In [ ]:
fig = shuffle_violin_plot(metric="most_likely", groups=["Human", "AI"])

### 8.2  Expected score: Original (matched) vs Shuffled

In [ ]:
fig = shuffle_violin_plot(metric="expected_score", groups=["Human", "AI"])

### 8.3  Numerical summary: bias before and after shuffling

In [ ]:
# Compare: matched-pair bias (Ollama) vs shuffled-pair bias (forward pass)
groups_cmp = ["AI", "Human"]

rows = []
for metric, col_name in [("most_likely", "most_likely"), ("expected_score", "expected_score")]:
    # Original Ollama
    means_orig = {g: orig_df[orig_df["group"] == g]["accuracy_score"].mean() for g in groups_cmp}
    # Shuffled forward pass
    means_shuf = {g: shuf_df[shuf_df["group"] == g][col_name].mean() for g in groups_cmp}

    orig_bias = means_orig.get("AI", np.nan) - means_orig.get("Human", np.nan)
    shuf_bias = means_shuf.get("AI", np.nan) - means_shuf.get("Human", np.nan)
    bias_change_pct = ((shuf_bias - orig_bias) / abs(orig_bias) * 100
                       if abs(orig_bias) > 1e-8 else np.nan)

    rows.append({
        "metric":      metric,
        "orig_AI":     round(means_orig.get("AI", np.nan), 3),
        "orig_Human":  round(means_orig.get("Human", np.nan), 3),
        "orig_bias":   round(orig_bias, 3),
        "shuf_AI":     round(means_shuf.get("AI", np.nan), 3),
        "shuf_Human":  round(means_shuf.get("Human", np.nan), 3),
        "shuf_bias":   round(shuf_bias, 3),
        "bias_change%": round(bias_change_pct, 1) if not np.isnan(bias_change_pct) else "N/A",
    })

summary_df = pd.DataFrame(rows).set_index("metric")
print("Bias (AI mean − Human mean) across conditions:")
print(summary_df.to_string())
print()
print("Interpretation:")
print("  If shuf_bias ≈ orig_bias  → bias is style-driven (survives question mismatch)")
print("  If shuf_bias ≈ 0          → bias requires content match; quality-driven")

### 8.4 Tertile analysis by response length — Original vs Shuffled

Original scores classified by the **matched** response length.  
Shuffle scores classified by the **shuffled** response length (not the question’s own response).  
Both use the same global tertile boundaries so results are directly comparable.

In [ ]:
# ── Compute tertile labels for original (α=0) and shuffle conditions ──────────
# Original condition: Llama forward pass at α=0 (matched pairs, no steering).
# Uses build_tertile_df from section 8.5 so tertile = matched response length.
# Shuffle condition:  Llama forward pass on shuffled pairs.
# Shuffle tertile = length of the *shuffled* response.

_ORIG_PROBE  = "authorship"
_ORIG_DIR    = "elasticnet"
_ORIG_LAYER  = 15

_orig_a0 = build_tertile_df(_ORIG_PROBE, _ORIG_DIR, _ORIG_LAYER, 0.0, asym_smp_df)
# _orig_a0 columns: group, argmax_score, expected_score, tertile, tertile_label

# Shuffle: tertile based on the *shuffled* response word count
all_resp_lengths = np.array([len(ds[i]["response"].split()) for i in range(len(ds))])
try:
    _q33, _q66 = q33, q66
except NameError:
    _q33, _q66 = np.percentile(all_resp_lengths, [33.33, 66.67])

def _tertile(lengths):
    t = np.zeros(len(lengths), dtype=int)
    t[lengths > _q33] = 1
    t[lengths > _q66] = 2
    return t

_shuf_ext = shuf_df.copy()
_shuf_ext["shuf_resp_len"] = _shuf_ext["response"].str.split().str.len()
_shuf_ext["tertile"]       = _tertile(_shuf_ext["shuf_resp_len"].values)

print(f"Tertile bounds (words): T1 ≤{_q33:.0f}  |  T2 {_q33:.0f}–{_q66:.0f}  |  T3 >{_q66:.0f}")
print("\nα=0 (original) tertile counts:")
print(_orig_a0.groupby(["group", "tertile"]).size().unstack(fill_value=0))
print("\nShuffle tertile counts (by shuffled response length):")
print(_shuf_ext.groupby(["group", "tertile"]).size().unstack(fill_value=0))


In [ ]:
def shuffle_tertile_plot(
    metric  = "most_likely",
    groups  = ["Human", "AI"],
    height  = 400,
    width   = 800,
):
    """α=0 (matched pairs) vs Shuffled by response-length tertile.

    Original tertile  = matched response length (via build_tertile_df).
    Shuffle tertile   = shuffled response length.
    """
    # Column names differ between the two DataFrames
    ORIG_COL = {"most_likely": "argmax_score",  "expected_score": "expected_score"}.get(metric, metric)
    SHUF_COL = {"most_likely": "most_likely",   "expected_score": "expected_score"}.get(metric, metric)
    metric_label = {
        "most_likely":    "Score (argmax, 1–7)",
        "expected_score": "Expected score Σ d·P(d)",
    }.get(metric, metric)

    TERT_LBL = ["T1 (Short)", "T2 (Medium)", "T3 (Long)"]

    def xpos(t_idx, is_shuffle):
        return t_idx * 2.2 + (0.6 if is_shuffle else 0.0)

    tick_vals  = [xpos(t, False) + 0.3 for t in range(3)]
    tick_texts = TERT_LBL

    fig = make_subplots(
        rows=1, cols=len(groups),
        subplot_titles=[f"{g} responses" for g in groups],
        horizontal_spacing=0.08, shared_yaxes=True,
    )

    cell_data = {}
    pv_infos  = []
    for ci, grp in enumerate(groups):
        for t_idx in range(3):
            orig_sub  = _orig_a0[(_orig_a0["group"] == grp) & (_orig_a0["tertile"] == t_idx)]
            orig_vals = orig_sub[ORIG_COL].dropna().values

            shuf_sub  = _shuf_ext[(_shuf_ext["group"] == grp) & (_shuf_ext["tertile"] == t_idx)]
            shuf_vals = shuf_sub[SHUF_COL].dropna().values

            cell_data[(ci, t_idx, "orig")] = orig_vals
            cell_data[(ci, t_idx, "shuf")] = shuf_vals
            p = mw_pvalue(orig_vals, shuf_vals)
            if not np.isnan(p):
                pv_infos.append({"ci": ci, "t_idx": t_idx, "p_raw": p})

    if pv_infos:
        _, pv_adj, _, _ = multipletests(
            [x["p_raw"] for x in pv_infos], alpha=0.05, method="fdr_bh")
        for i, info in enumerate(pv_infos):
            info["p_adj"] = pv_adj[i]
    pv_lookup = {(x["ci"], x["t_idx"]): x.get("p_adj", x["p_raw"]) for x in pv_infos}

    shown = set()
    for ci, grp in enumerate(groups):
        col_n = ci + 1
        xref  = "x" if col_n == 1 else f"x{col_n}"
        yref  = "y" if col_n == 1 else f"y{col_n}"
        color = GROUP_COLORS.get(grp, "#999999")

        for t_idx in range(3):
            for is_shuf, key, lbl_suffix in [
                (False, "orig", "α=0 (matched)"),
                (True,  "shuf", "shuffled"),
            ]:
                vals     = cell_data[(ci, t_idx, key)]
                opacity  = 0.75 if is_shuf else 0.30
                lbl      = f"{grp} — {lbl_suffix}"
                show_leg = lbl not in shown
                shown.add(lbl)
                xv = xpos(t_idx, is_shuf)

                fig.add_trace(go.Violin(
                    y=vals, name=lbl, x0=xv,
                    fillcolor=color, opacity=opacity,
                    box_visible=True, meanline_visible=True, width=0.5,
                    showlegend=show_leg, legendgroup=lbl,
                    line=dict(width=2, color="black"),
                    marker=dict(color=color, line=dict(color="black", width=1)),
                    scalemode="width",
                ), row=1, col=col_n)

                fig.add_annotation(
                    x=xv, y=9.05, text=f"n={len(vals)}",
                    showarrow=False,
                    font=dict(size=11, family="Arial", color="black"),
                    xref=xref, yref=yref,
                )

        for t_idx in range(3):
            p = pv_lookup.get((ci, t_idx))
            if p is not None:
                add_bracket(fig, xpos(t_idx, False), xpos(t_idx, True),
                            7.8, fmt_p(p), xref, yref)

    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
        title=dict(
            text=f"α=0 (matched) vs Shuffled — by response length tertile | {metric}",
            font=dict(size=20, family="Arial"),
            y=0.98,
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
            font=dict(size=12, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=100, b=80, l=100, r=60),
    )

    for ci, grp in enumerate(groups):
        col_n = ci + 1
        yax = "yaxis"  if col_n == 1 else f"yaxis{col_n}"
        xax = "xaxis"  if col_n == 1 else f"xaxis{col_n}"
        fig.layout[yax].update(
            title=dict(text=metric_label if col_n == 1 else "",
                       font=dict(size=26, family="Arial", color="black")),
            range=[0, 9.5], showgrid=False, zeroline=False,
            tickfont=dict(size=24, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        )
        fig.layout[xax].update(
            ticktext=tick_texts, tickvals=tick_vals,
            showgrid=False, zeroline=False,
            tickfont=dict(size=18, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
            range=[-0.4, 5.4],
        )
    for ann in fig.layout.annotations[:len(groups)]:
        ann.font.size   = 26
        ann.font.family = "Arial"
        ann.font.color  = "black"

    fig.show()
    fig.write_image(f"{MODEL_KEY}_shuffle_tertile_violin_{metric}.svg", scale=3)
    fig
    return fig


for metric in ["most_likely", "expected_score"]:
    shuffle_tertile_plot(metric=metric)


In [ ]:
def shuffle_tertile_plot_combined(
    metric  = "most_likely",
    height  = 400,
    width   = 500,
):
    ORIG_COL = {"most_likely": "argmax_score", "expected_score": "expected_score"}.get(metric, metric)
    SHUF_COL = {"most_likely": "most_likely",  "expected_score": "expected_score"}.get(metric, metric)
    metric_label = {
        "most_likely":    "Score (argmax, 1–7)",
        "expected_score": "Expected score Σ d·P(d)",
    }.get(metric, metric)

    TERT_LBL = ["T1 (Short)", "T2 (Medium)", "T3 (Long)"]
    COLOR    = "#8F055A"

    def xpos(t_idx, is_shuffle):
        return t_idx * 2.2 + (0.6 if is_shuffle else 0.0)

    tick_vals  = [xpos(t, False) + 0.3 for t in range(3)]

    fig = go.Figure()

    cell_data = {}
    pv_infos  = []
    for t_idx in range(3):
        orig_vals = _orig_a0[_orig_a0["tertile"] == t_idx][ORIG_COL].dropna().values
        shuf_vals = _shuf_ext[_shuf_ext["tertile"] == t_idx][SHUF_COL].dropna().values
        cell_data[(t_idx, "orig")] = orig_vals
        cell_data[(t_idx, "shuf")] = shuf_vals
        p = wil_pvalue(orig_vals, shuf_vals)
        if not np.isnan(p):
            pv_infos.append({"t_idx": t_idx, "p_raw": p})

    if pv_infos:
        _, pv_adj, _, _ = multipletests(
            [x["p_raw"] for x in pv_infos], alpha=0.05, method="fdr_bh")
        for i, info in enumerate(pv_infos):
            info["p_adj"] = pv_adj[i]
    pv_lookup = {x["t_idx"]: x.get("p_adj", x["p_raw"]) for x in pv_infos}

    shown = set()
    for t_idx in range(3):
        for is_shuf, key, lbl_suffix, opacity in [
            (False, "orig", "α=0 (matched)", 0.30),
            (True,  "shuf", "shuffled",      0.75),
        ]:
            vals     = cell_data[(t_idx, key)]
            lbl      = lbl_suffix
            show_leg = lbl not in shown
            shown.add(lbl)
            xv = xpos(t_idx, is_shuf)

            fig.add_trace(go.Violin(
                y=vals, name=lbl, x0=xv,
                fillcolor=COLOR, opacity=opacity,
                box_visible=True, meanline_visible=True, width=0.5,
                showlegend=show_leg, legendgroup=lbl,
                line=dict(width=2, color="black"),
                marker=dict(color=COLOR, line=dict(color="black", width=1)),
                scalemode="width",
            ))

            fig.add_annotation(
                x=xv, y=9.05, text=f"n={len(vals)}",
                showarrow=False,
                font=dict(size=13, family="Arial", color="black"),
            )

        p = pv_lookup.get(t_idx)
        if p is not None:
            add_bracket(fig, xpos(t_idx, False), xpos(t_idx, True),
                        7.8, fmt_p(p), "x", "y")

    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
        title=dict(
            text=f"α=0 (matched) vs Shuffled — by tertile | {metric}",
            font=dict(size=18, family="Arial"),
            y=0.98,
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
            font=dict(size=14, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=100, b=80, l=100, r=60),
        xaxis=dict(
            ticktext=TERT_LBL, tickvals=tick_vals,
            showgrid=False, zeroline=False,
            tickfont=dict(size=18, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
            range=[-0.4, 5.4],
        ),
        yaxis=dict(
            title=dict(text=metric_label, font=dict(size=22, family="Arial", color="black")),
            range=[0, 9.5], showgrid=False, zeroline=False,
            tickfont=dict(size=20, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
        ),
    )

    fig.show()
    fig.write_image(f"{MODEL_KEY}_shuffle_tertile_combined_{metric}.svg", scale=3)
    return fig

for metric in ["most_likely", "expected_score"]:
    shuffle_tertile_plot_combined(metric=metric)


In [ ]:
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from scipy.stats import linregress

def _p_trend(vals_by_tertile):
    ranks = np.concatenate([np.full(len(vals_by_tertile[t]), t + 1) for t in range(3)])
    vals  = np.concatenate([vals_by_tertile[t] for t in range(3)])
    return linregress(ranks, vals).pvalue


def fmt_ptrend(p):
    if p < 0.001: return "p-trend < 0.001"
    if p < 0.01:  return f"p-trend = {p:.3f}"
    return f"p-trend = {p:.2f}"

def shuffle_tertile_plot_combined(metric="most_likely", height=400, width=500):
    ORIG_COL = {"most_likely": "argmax_score", "expected_score": "expected_score"}.get(metric, metric)
    SHUF_COL = {"most_likely": "most_likely",  "expected_score": "expected_score"}.get(metric, metric)
    metric_label = {
        "most_likely":    "Score (argmax, 1–7)",
        "expected_score": "Expected score Σ d·P(d)",
    }.get(metric, metric)

    TERT_LBL = ["T1 (Short)", "T2 (Medium)", "T3 (Long)"]
    COLOR    = "#8F055A"

    def xpos(t_idx, is_shuffle):
        return t_idx * 2.2 + (0.6 if is_shuffle else 0.0)

    tick_vals = [xpos(t, False) + 0.3 for t in range(3)]
    fig = go.Figure()

    cell_data = {}
    for t_idx in range(3):
        cell_data[(t_idx, "orig")] = _orig_a0[_orig_a0["tertile"] == t_idx][ORIG_COL].dropna().values
        cell_data[(t_idx, "shuf")] = _shuf_ext[_shuf_ext["tertile"] == t_idx][SHUF_COL].dropna().values

    p_trend_orig = _p_trend({t: cell_data[(t, "orig")] for t in range(3)})
    p_trend_shuf = _p_trend({t: cell_data[(t, "shuf")] for t in range(3)})

    shown = set()
    for t_idx in range(3):
        for is_shuf, key, lbl_suffix, opacity in [
            (False, "orig", "Matched", 0.30),
            (True,  "shuf", "Shuffled",      0.75),
        ]:
            vals     = cell_data[(t_idx, key)]
            lbl      = lbl_suffix
            show_leg = lbl not in shown
            shown.add(lbl)
            xv = xpos(t_idx, is_shuf)

            fig.add_trace(go.Violin(
                y=vals, name=lbl, x0=xv,
                fillcolor=COLOR, opacity=opacity,
                box_visible=True, meanline_visible=True, width=0.5,
                showlegend=show_leg, legendgroup=lbl,
                line=dict(width=2, color="black"),
                marker=dict(color=COLOR, line=dict(color="black", width=1)),
                scalemode="width",
            ))

    # p-trend orig — línea continua
    x0o, x1o = xpos(0, False), xpos(2, False)
    fig.add_shape(type="line", x0=x0o, x1=x1o, y0=8.6, y1=8.6,
                  line=dict(color="black", width=1.5))
    fig.add_annotation(x=(x0o + x1o) / 2, y=8.82,
                       text=f"Matched: {fmt_ptrend(p_trend_orig)}",
                       showarrow=False,
                       font=dict(size=14, family="Arial", color="black"))

    # p-trend shuffled — línea punteada
    x0s, x1s = xpos(0, True), xpos(2, True)
    fig.add_shape(type="line", x0=x0s, x1=x1s, y0=7.6, y1=7.6,
                  line=dict(color="black", width=1.5, dash="dot"))
    fig.add_annotation(x=(x0s + x1s) / 2, y=7.83,
                       text=f"Shuffled: {fmt_ptrend(p_trend_shuf)}",
                       showarrow=False,
                       font=dict(size=14, family="Arial", color="black"))

    fig.update_layout(
        font=dict(size=24, family="Arial", color="black"),
        height=height, width=width,
        template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
            font=dict(size=14, family="Arial"),
            bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=2,
        ),
        margin=dict(t=80, b=80, l=100, r=60),
        xaxis=dict(
            ticktext=TERT_LBL, tickvals=tick_vals,
            showgrid=False, zeroline=False,
            tickfont=dict(size=18, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
            range=[-0.4, 5.4],
        ),
        yaxis=dict(
            title=dict(text=metric_label, font=dict(size=22, family="Arial", color="black")),
            range=[0, 9.5], showgrid=False, zeroline=False,
            tickfont=dict(size=20, family="Arial", color="black"),
            showline=True, linewidth=2, linecolor="black", mirror=False,
            tickvals=[1, 2, 3, 4, 5, 6, 7],
        ),
    )

    fig.show()
    fig.write_image(f"{MODEL_KEY}_shuffle_tertile_combined_{metric}.svg", scale=3)
    return fig

_shuf_ext = shuf_wt_df
for metric in ["most_likely", "expected_score"]:
    shuffle_tertile_plot_combined(metric=metric)


In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import numpy as np

THRESHOLD = 4
BAR_WIDTH = 0.32

COLOR_ORIG_HI = "#2166AC"; COLOR_ORIG_LO = "#92C5DE"
COLOR_SHUF_HI = "#8F055A"; COLOR_SHUF_LO = "#D4A0C0"

orig_hi, orig_lo, shuf_hi, shuf_lo = [], [], [], []
n_orig_hi, n_orig_lo, n_shuf_hi, n_shuf_lo = [], [], [], []

for t_idx in range(3):
    ov = _orig_a0[_orig_a0["tertile"] == t_idx]["argmax_score"].dropna().values
    sv = shuf_wt_df[shuf_wt_df["tertile"] == t_idx]["most_likely"].dropna().values
    orig_hi.append(100.0 * (ov >= THRESHOLD).mean())
    orig_lo.append(100.0 * (ov  < THRESHOLD).mean())
    shuf_hi.append(100.0 * (sv >= THRESHOLD).mean())
    shuf_lo.append(100.0 * (sv  < THRESHOLD).mean())
    n_orig_hi.append((ov >= THRESHOLD).sum()); n_orig_lo.append((ov < THRESHOLD).sum())
    n_shuf_hi.append((sv >= THRESHOLD).sum()); n_shuf_lo.append((sv < THRESHOLD).sum())

deltas = [orig_hi[t] - shuf_hi[t] for t in range(3)]

p_raws = []
for t_idx in range(3):
    table = [[n_orig_hi[t_idx], n_orig_lo[t_idx]],
             [n_shuf_hi[t_idx], n_shuf_lo[t_idx]]]
    _, p = fisher_exact(table)
    p_raws.append(p)
_, p_adj, _, _ = multipletests(p_raws, alpha=0.05, method="fdr_bh")

def _fmt_p(p):
    if p < 0.001: return "p < 0.001"
    return f"p = {p:.3f}"

def _fmt_delta(d):
    sign = "+" if d >= 0 else ""
    return f"Δ={sign}{d:.1f}%"

x = np.arange(3)
fig = go.Figure()

fig.add_trace(go.Bar(x=x - BAR_WIDTH/2, y=orig_lo, name="Matched, score < 4",
    marker_color=COLOR_ORIG_LO, marker_line=dict(color="black", width=0.8),
    width=BAR_WIDTH, legendgroup="orig"))
fig.add_trace(go.Bar(x=x - BAR_WIDTH/2, y=orig_hi, name="Matched, score ≥ 4",
    marker_color=COLOR_ORIG_HI, marker_line=dict(color="black", width=0.8),
    width=BAR_WIDTH, legendgroup="orig",
    text=[f"{p:.0f}%" for p in orig_hi],
    textposition="inside", insidetextanchor="middle",
    textfont=dict(size=16, family="Arial", color="white")))

fig.add_trace(go.Bar(x=x + BAR_WIDTH/2, y=shuf_lo, name="Shuffled, score < 4",
    marker_color=COLOR_SHUF_LO, marker_line=dict(color="black", width=0.8),
    width=BAR_WIDTH, legendgroup="shuf"))
fig.add_trace(go.Bar(x=x + BAR_WIDTH/2, y=shuf_hi, name="Shuffled, score ≥ 4",
    marker_color=COLOR_SHUF_HI, marker_line=dict(color="black", width=0.8),
    width=BAR_WIDTH, legendgroup="shuf",
    text=[f"{p:.0f}%" for p in shuf_hi],
    textposition="inside", insidetextanchor="middle",
    textfont=dict(size=16, family="Arial", color="white")))

for t_idx in range(3):
    x0 = t_idx - BAR_WIDTH / 2
    x1 = t_idx + BAR_WIDTH / 2
    y_bar, dy = 102, 3
    for xi in [x0, x1]:
        fig.add_shape(type="line", x0=xi, x1=xi, y0=y_bar, y1=y_bar + dy,
                      line=dict(color="black", width=1))
    fig.add_shape(type="line", x0=x0, x1=x1, y0=y_bar + dy, y1=y_bar + dy,
                  line=dict(color="black", width=1))
    fig.add_annotation(x=t_idx, y=y_bar + dy + 2.5,
                       text=_fmt_p(p_adj[t_idx]), showarrow=False,
                       font=dict(size=9, family="Arial", color="grey"))
    fig.add_annotation(x=t_idx, y=y_bar + dy + 9,
                       text=_fmt_delta(deltas[t_idx]), showarrow=False,
                       font=dict(size=14, family="Arial", color="black"))

fig.update_layout(
    barmode="stack",
    font=dict(size=14, family="Arial", color="black"),
    height=340, width=450,
    template="simple_white", plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="v", x=1.02, y=1.0, xanchor="left", yanchor="top",
                font=dict(size=12, family="Arial"), bgcolor="rgba(255,255,255,0)",
                borderwidth=0, tracegroupgap=2, itemwidth=30),
    margin=dict(t=90, b=50, l=52, r=140),
    xaxis=dict(
        ticktext=["T1<br><sup>Short</sup>", "T2<br><sup>Medium</sup>", "T3<br><sup>Long</sup>"],
        tickvals=list(x), showgrid=False, zeroline=False,
        tickfont=dict(size=14, family="Arial", color="black"),
        showline=True, linewidth=1, linecolor="black",
        ticks="outside", ticklen=3),
    yaxis=dict(
        title=dict(text="Responses (%)", font=dict(size=12, family="Arial")),
        range=[0, 122], showgrid=False, zeroline=False,
        tickfont=dict(size=14, family="Arial", color="black"),
        showline=True, linewidth=1, linecolor="black",
        ticks="outside", ticklen=3, tickvals=[0, 25, 50, 75, 100]),
)

fig.show()
fig.write_image(f"{MODEL_KEY}_stacked_pct_tertile_delta.svg", scale=3)


In [ ]:
from scipy.stats import linregress, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import itertools

def _fmt_ptrend(p):
    if p < 0.001: return "p-trend < 0.001"
    if p < 0.01:  return f"p-trend = {p:.3f}"
    return f"p-trend = {p:.2f}"

def _fmt_p(p):
    if p < 0.001: return "p < 0.001"
    return f"p = {p:.3f}"

def _bracket(fig, x0, x1, y, text, dy=0.18):
    for xi in [x0, x1]:
        fig.add_shape(type="line", x0=xi, x1=xi, y0=y, y1=y+dy,
                      line=dict(color="black", width=1.5))
    fig.add_shape(type="line", x0=x0, x1=x1, y0=y+dy, y1=y+dy,
                  line=dict(color="black", width=1.5))
    fig.add_annotation(x=(x0+x1)/2, y=y+dy+0.12, text=text,
                       showarrow=False,
                       font=dict(size=16, family="Arial", color="black"))

DELTA_COL = "delta_expected_score"
TERT_LBL  = ["T1 (Short)", "T2 (Medium)", "T3 (Long)"]
COLOR     = "#8F055A"

dvals = {t: shuf_wt_df[shuf_wt_df["tertile"] == t][DELTA_COL].dropna().values
         for t in range(3)}

ranks_all = np.concatenate([np.full(len(dvals[t]), t + 1) for t in range(3)])
vals_all  = np.concatenate([dvals[t] for t in range(3)])
p_trend   = linregress(ranks_all, vals_all).pvalue

pairs  = list(itertools.combinations(range(3), 2))
p_raws = []
for a, b in pairs:
    _, p = mannwhitneyu(dvals[a], dvals[b], alternative="two-sided")
    p_raws.append(p)
_, p_adj, _, _ = multipletests(p_raws, alpha=0.05, method="fdr_bh")

ymin = min(v.min() for v in dvals.values()) - 0.5
ymax = max(v.max() for v in dvals.values()) + 1.5

fig = go.Figure()

for t_idx in range(3):
    fig.add_trace(go.Violin(
        y=dvals[t_idx], x0=float(t_idx),
        fillcolor=COLOR, opacity=0.65,
        box_visible=True, meanline_visible=True, width=0.55,
        showlegend=False,
        line=dict(width=2, color="black"),
        marker=dict(color=COLOR, line=dict(color="black", width=1)),
        scalemode="width",
    ))

bracket_ys = [ymax + 0.6, ymax + 1.4, ymax + 2.2]
for i, ((a, b), padj) in enumerate(zip(pairs, p_adj)):
    _bracket(fig, float(a), float(b), bracket_ys[i], _fmt_p(padj))

fig.add_annotation(
    x=1.0, y=bracket_ys[-1] + 0.6,
    text=_fmt_ptrend(p_trend),
    showarrow=False,
    font=dict(size=18, family="Arial", color="black"),
)

fig.update_layout(
    font=dict(size=24, family="Arial", color="black"),
    height=500, width=500,
    template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=120, b=80, l=100, r=60),
    xaxis=dict(
        ticktext=TERT_LBL, tickvals=[0, 1, 2],
        showgrid=False, zeroline=False,
        tickfont=dict(size=18, family="Arial", color="black"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
        range=[-0.5, 2.5],
    ),
    yaxis=dict(
        title=dict(text="Δ Score (matched − shuffled)", font=dict(size=22, family="Arial", color="black")),
        range=[ymin, bracket_ys[-1] + 1.2],
        showgrid=False, zeroline=False,
        tickfont=dict(size=20, family="Arial", color="black"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
    ),
)

fig.show()
fig.write_image(f"{MODEL_KEY}_delta_by_tertile.svg", scale=3)


In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

with open(TABLE_DIR / f"{MODEL_KEY}_cross_tertile_shuffle.json") as f:
    ct_data = json.load(f)
ct_df = pd.DataFrame(ct_data["records"])

CONDITIONS = ["T1→T3", "T3→T1", "T2→T1", "T2→T3"]
COND_LBL = [
    "T1 q's<br><sup>orig T1 resp → T3 resp</sup>",
    "T3 q's<br><sup>orig T3 resp → T1 resp</sup>",
    "T2 q's<br><sup>orig T2 resp → T1 resp</sup>",
    "T2 q's<br><sup>orig T2 resp → T3 resp</sup>",
]
_TERT_IDX = {"T1": 0, "T2": 1, "T3": 2}

# metric: "expected_score" o "most_likely"
METRIC      = "expected_score"
ORIG_COL    = {"most_likely": "argmax_score", "expected_score": "expected_score"}[METRIC]
CROSS_COL   = "cross_expected_score"   # siempre expected_score del cross
METRIC_LBL  = {"most_likely": "Score (argmax, 1–7)",
                "expected_score": "Expected score Σ d·P(d)"}[METRIC]

COLOR = "#8F055A"

def xpos(c_idx, is_cross):
    return c_idx * 2.2 + (0.6 if is_cross else 0.0)

tick_vals = [xpos(c, False) + 0.3 for c in range(len(CONDITIONS))]

def _fmt_p(p):
    if p < 0.001: return "p < 0.001"
    return f"p = {p:.3f}"

def _add_bracket(fig, x0, x1, y, text, dy=0.15):
    for xi in [x0, x1]:
        fig.add_shape(type="line", x0=xi, x1=xi, y0=y, y1=y+dy,
                      line=dict(color="black", width=1.5))
    fig.add_shape(type="line", x0=x0, x1=x1, y0=y+dy, y1=y+dy,
                  line=dict(color="black", width=1.5))
    fig.add_annotation(x=(x0+x1)/2, y=y+dy+0.1, text=text,
                       showarrow=False,
                       font=dict(size=14, family="Arial", color="black"))

# ── Collect data ──────────────────────────────────────────────────────────────
cell_data = {}
pv_raws   = []
for c_idx, cond in enumerate(CONDITIONS):
    q_tert = cond.split("→")[0]                         # "T1", "T2", "T3"
    t_idx  = _TERT_IDX[q_tert]

    # Originales: α=0 de steering, mismo tertil que la pregunta
    orig_v  = _orig_a0[_orig_a0["tertile"] == t_idx][ORIG_COL].dropna().values

    # Cross: respuestas del tertil contrario (del JSON de cross-tertile)
    cross_v = ct_df[ct_df["condition"] == cond]["cross_expected_score"].dropna().values

    cell_data[(c_idx, "orig")]  = orig_v
    cell_data[(c_idx, "cross")] = cross_v

    try:
        _, p = mannwhitneyu(orig_v, cross_v, alternative="two-sided")
    except Exception:
        p = float("nan")
    pv_raws.append(p)

# FDR
valid = [(i, p) for i, p in enumerate(pv_raws) if not np.isnan(p)]
if valid:
    idxs, pvals = zip(*valid)
    _, padj, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
    pv_adj = {i: pa for i, pa in zip(idxs, padj)}
else:
    pv_adj = {}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig   = go.Figure()
shown = set()

for c_idx, cond in enumerate(CONDITIONS):
    q_tert = cond.split("→")[0]
    r_tert = cond.split("→")[1]
    for is_cross, key, lbl, opacity in [
        (False, "orig",  f"Original ({q_tert} resp)", 0.30),
        (True,  "cross", f"Swapped ({r_tert} resp)",  0.75),
    ]:
        vals     = cell_data[(c_idx, key)]
        show_leg = lbl not in shown
        shown.add(lbl)
        xv = xpos(c_idx, is_cross)

        fig.add_trace(go.Violin(
            y=vals, name=lbl, x0=xv,
            fillcolor=COLOR, opacity=opacity,
            box_visible=True, meanline_visible=True, width=0.5,
            showlegend=show_leg, legendgroup=key,
            line=dict(width=2, color="black"),
            marker=dict(color=COLOR, line=dict(color="black", width=1)),
            scalemode="width",
        ))
        fig.add_annotation(
            x=xv, y=9.1, text=f"n={len(vals)}",
            showarrow=False,
            font=dict(size=13, family="Arial", color="black"),
        )

    p = pv_adj.get(c_idx)
    if p is not None:
        _add_bracket(fig, xpos(c_idx, False), xpos(c_idx, True), 7.8, _fmt_p(p))

fig.update_layout(
    font=dict(size=24, family="Arial", color="black"),
    height=480, width=900,
    template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(
        orientation="h", yanchor="bottom", y=1.06, xanchor="center", x=0.5,
        font=dict(size=13, family="Arial"),
        bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=2,
    ),
    margin=dict(t=80, b=110, l=100, r=60),
    xaxis=dict(
        ticktext=COND_LBL, tickvals=tick_vals,
        showgrid=False, zeroline=False,
        tickfont=dict(size=13, family="Arial", color="black"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
        range=[-0.4, len(CONDITIONS) * 2.2 - 0.6],
    ),
    yaxis=dict(
        title=dict(text=METRIC_LBL, font=dict(size=22, family="Arial", color="black")),
        range=[0, 9.8], showgrid=False, zeroline=False,
        tickfont=dict(size=20, family="Arial", color="black"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
        tickvals=[1, 2, 3, 4, 5, 6, 7],
    ),
)

fig.show()
fig.write_image(f"{MODEL_KEY}_cross_tertile_violin_{METRIC}.svg", scale=3)


In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

with open(TABLE_DIR / f"{MODEL_KEY}_cross_tertile_shuffle.json") as f:
    ct_data = json.load(f)
ct_df = pd.DataFrame(ct_data["records"])

# delta = orig_expected_score - cross_expected_score  (ya está en el JSON)
DELTA_COL = "delta_expected_score"

CONDITIONS = ["T1→T3", "T3→T1", "T2→T1", "T2→T3"]
COND_LBL = [
    "T1→T3<br><sup>short→long</sup>",
    "T3→T1<br><sup>long→short</sup>",
    "T2→T1<br><sup>medium→short</sup>",
    "T2→T3<br><sup>medium→long</sup>",
]
COLOR = "#8F055A"

def _fmt_p(p):
    if p < 0.001: return "p < 0.001"
    return f"p = {p:.3f}"

def _add_bracket(fig, x, y, text, dy=0.18):
    fig.add_shape(type="line", x0=x-0.25, x1=x+0.25, y0=y, y1=y,
                  line=dict(color="black", width=1.5))
    fig.add_annotation(x=x, y=y+dy, text=text,
                       showarrow=False,
                       font=dict(size=14, family="Arial", color="black"))

# ── Collect deltas + Wilcoxon vs 0 ───────────────────────────────────────────
dvals   = {}
pv_raws = []
for c_idx, cond in enumerate(CONDITIONS):
    sub = ct_df[ct_df["condition"] == cond][DELTA_COL].dropna().values
    dvals[c_idx] = sub
    try:
        # one-sample Wilcoxon signed-rank vs 0
        _, p = wilcoxon(sub, alternative="two-sided")
    except Exception:
        p = float("nan")
    pv_raws.append(p)

# FDR
valid = [(i, p) for i, p in enumerate(pv_raws) if not np.isnan(p)]
if valid:
    idxs, pvals = zip(*valid)
    _, padj, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
    pv_adj = {i: pa for i, pa in zip(idxs, padj)}
else:
    pv_adj = {}

# ── Plot ──────────────────────────────────────────────────────────────────────
ymin = min(v.min() for v in dvals.values()) - 0.3
ymax = max(v.max() for v in dvals.values()) + 1.2

fig = go.Figure()

for c_idx in range(len(CONDITIONS)):
    vals = dvals[c_idx]
    m    = float(np.mean(vals))
    fig.add_trace(go.Violin(
        y=vals, x0=float(c_idx),
        fillcolor=COLOR, opacity=0.70,
        box_visible=True, meanline_visible=True, width=0.55,
        showlegend=False,
        line=dict(width=2, color="black"),
        marker=dict(color=COLOR, line=dict(color="black", width=1)),
        scalemode="width",
    ))
    # mean label
    fig.add_annotation(
        x=float(c_idx), y=ymax - 0.05,
        text=f"mean={m:+.2f}",
        showarrow=False,
        font=dict(size=13, family="Arial", color="black"),
    )
    # p-value (Wilcoxon vs 0)
    p = pv_adj.get(c_idx)
    if p is not None:
        _add_bracket(fig, float(c_idx), ymax + 0.35, _fmt_p(p))

# Reference line at 0
fig.add_shape(
    type="line", x0=-0.5, x1=len(CONDITIONS) - 0.5, y0=0, y1=0,
    line=dict(color="black", width=1.5, dash="dash"),
)

fig.update_layout(
    font=dict(size=24, family="Arial", color="black"),
    autosize=False,
    height=260, width=300,
    template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
    
    xaxis=dict(
        ticktext=COND_LBL, tickvals=list(range(len(CONDITIONS))),
        showgrid=False, zeroline=False,
        tickfont=dict(size=14, family="Arial", color="black"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
        range=[-0.6, len(CONDITIONS) - 0.4],
    ),
    yaxis=dict(
        title=dict(text="Delta score (orig − swapped)",
                   font=dict(size=20, family="Arial", color="black")),
        range=[ymin, ymax + 1.0],
        showgrid=False, zeroline=False,
        tickfont=dict(size=20, family="Arial", color="black"),
        showline=True, linewidth=2, linecolor="black", mirror=False,
    ),
)

fig.show(renderer="iframe")

fig.write_image(f"{MODEL_KEY}_cross_tertile_delta_violin.svg", scale=3)


In [ ]:
import h5py
import numpy as np
import plotly.graph_objects as go

H5_PATH       = "../../../results/hidden_states/llama-3.1-8b.h5"
FIXED_OVERHEAD = 80  # 75 tokens JSON template tail + 5 assistant prefix

with h5py.File(H5_PATH, "r") as f:
    score_pos  = f["sample_metadata"]["score_token_pos"][:]
    resp_start = f["sample_metadata"]["response_start_pos"][:]

resp_tokens = (score_pos - resp_start - FIXED_OVERHEAD).astype(int)

# Tertile boundaries
q33 = float(np.percentile(resp_tokens, 33.33))
q66 = float(np.percentile(resp_tokens, 66.67))

tertile = np.where(resp_tokens <= q33, 0, np.where(resp_tokens <= q66, 1, 2))

TERT_LABELS = [f"T1 · Short<br>(≤{q33:.0f} tok)",
               f"T2 · Medium<br>({q33:.0f}–{q66:.0f} tok)",
               f"T3 · Long<br>(>{q66:.0f} tok)"]
TERT_COLORS = ["#4393C3", "#F4A582", "#D6604D"]
FONT = dict(family="Arial", color="black")

fig = go.Figure()

for t_idx in range(3):
    vals = resp_tokens[tertile == t_idx]
    fig.add_trace(go.Violin(
        y=vals,
        name=TERT_LABELS[t_idx],
        x0=t_idx,
        fillcolor=TERT_COLORS[t_idx],
        opacity=0.7,
        box_visible=True,
        meanline_visible=True,
        width=0.9,
        line=dict(color="black", width=1.5),
        marker=dict(color=TERT_COLORS[t_idx],
                    line=dict(color="black", width=0.8), size=3),
        scalemode="width",
        showlegend=True,
    ))
    

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(**FONT, size=23),
    title=dict(
        text="Response length by tertile (tokens, fixed overhead removed)",
        font=dict(**FONT, size=24), x=0.5, xanchor="center",
    ),
    xaxis=dict(
        tickvals=[0, 1, 2],
        ticktext=[f"T1<br>(≤{q33:.0f} tok)",
                  f"T2<br>({q33:.0f}–{q66:.0f} tok)",
                  f"T3<br>(>{q66:.0f} tok)"],
        tickfont=dict(**FONT, size=22),
        showgrid=False, zeroline=False,
        showline=True, linewidth=2, linecolor="black",
        mirror=False, range=[-0.6, 2.6],
    ),
    yaxis=dict(
        title=dict(text="Response length (tokens)", font=dict(**FONT, size=23)),
        tickfont=dict(**FONT, size=22),
        showgrid=False, zeroline=False,
        showline=True, linewidth=2.5, linecolor="black",
        mirror=False, ticks="outside", ticklen=4,
    ),
    showlegend=False,
    width=440, height=400,
    margin=dict(t=60, b=60, l=70, r=20),
)

fig.write_image("verbosity_tertile_violin_tokens.svg", scale=3)
fig.write_image("verbosity_tertile_violin_tokens.png", scale=3)
fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
from src.config import load_config
from src.data import load_cardio_dataset, get_unique_responses

# ── Data ──────────────────────────────────────────────────────────────────────
cfg = load_config("../../../config/config.yaml")
raw = load_cardio_dataset(cfg)
ds  = get_unique_responses(raw, target_evaluator="Llama3.1:8b")

rows = [{"source": r["response_source"], "length": len(r["response"].split())}
        for r in ds]

ai_lens    = [r["length"] for r in rows if r["source"] == "AI"]
human_lens = [r["length"] for r in rows if r["source"] == "Human"]
all_lens   = ai_lens + human_lens

q33 = float(np.percentile(all_lens, 33.33))
q66 = float(np.percentile(all_lens, 66.67))

FONT   = dict(family="Arial", color="black")
AI_COL  = "#2166AC"
HUM_COL = "#D6604D"
BINS    = dict(start=0, end=310, size=12)

fig = go.Figure()

# Tertile shading
for x0, x1, lbl in [
    (0,   q33, "T1 · Short"),
    (q33, q66, "T2 · Medium"),
    (q66, 310, "T3 · Long"),
]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor="#f5f5f5",
                  opacity=1.0, layer="below", line_width=0)
    fig.add_annotation(
        x=(x0 + x1) / 2, y=1.02, yref="paper",
        text=lbl, showarrow=False, xanchor="center",
        font=dict(size=11, family="Arial", color="#666666"),
    )

# Tertile boundary lines
for q in [q33, q66]:
    fig.add_vline(x=q, line=dict(color="#aaaaaa", width=1.2, dash="dash"))
    fig.add_annotation(
        x=q, y=0.96, yref="paper",
        text=f"{q:.0f} w", showarrow=False,
        xanchor="left", xshift=4,
        font=dict(size=10, family="Arial", color="#888888"),
    )

# Histograms
fig.add_trace(go.Histogram(
    x=human_lens,
    name=f"Human (n={len(human_lens)})",
    xbins=BINS, histnorm="probability density",
    opacity=0.5,
    marker=dict(color=HUM_COL, line=dict(color="white", width=0.3)),
))
fig.add_trace(go.Histogram(
    x=ai_lens,
    name=f"AI (n={len(ai_lens)})",
    xbins=BINS, histnorm="probability density",
    opacity=0.5,
    marker=dict(color=AI_COL, line=dict(color="white", width=0.3)),
))

# Median lines
for vals, color in [(human_lens, HUM_COL), (ai_lens, AI_COL)]:
    med = float(np.median(vals))
    fig.add_vline(x=med, line=dict(color=color, width=1.8, dash="dot"))
    fig.add_annotation(
        x=med, y=0.78, yref="paper",
        text=f"med={med:.0f}",
        showarrow=False, xanchor="left", xshift=4,
        font=dict(size=10, family="Arial", color=color),
    )

fig.update_layout(
    barmode="overlay",
    template="plotly_white",
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(**FONT, size=13),
    title=dict(
        text="Response length distribution — AI vs Human",
        font=dict(**FONT, size=14), x=0.5, xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="Response length (words)", font=dict(**FONT, size=13)),
        tickfont=dict(**FONT, size=12),
        showgrid=False, zeroline=False,
        showline=True, linewidth=1, linecolor="black",
        mirror=False, ticks="outside", ticklen=4,
        dtick=50, range=[0, 310],
    ),
    yaxis=dict(
        title=dict(text="Probability density", font=dict(**FONT, size=13)),
        tickfont=dict(**FONT, size=12),
        showgrid=False, zeroline=False,
        showline=True, linewidth=1, linecolor="black",
        mirror=False, ticks="outside", ticklen=4,
    ),
    legend=dict(
        x=0.97, y=0.97, xanchor="right", yanchor="top",
        bgcolor="rgba(255,255,255,0.9)", bordercolor="#cccccc", borderwidth=1,
        font=dict(**FONT, size=12),
    ),
    width=540, height=360,
    margin=dict(t=65, b=55, l=65, r=20),
)

fig.write_image("verbosity_distribution.svg", scale=3)
fig.write_image("verbosity_distribution.png", scale=3)
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.formula.api as smf
from src.config import load_config
from src.data import load_cardio_dataset

cfg = load_config("../../../config/config.yaml")
raw = load_cardio_dataset(cfg)
df  = raw.to_pandas()

df["resp_len"] = df["response"].str.split().str.len()
unique_lens    = df.drop_duplicates("response")["resp_len"].values
q33 = np.percentile(unique_lens, 33.33)
q66 = np.percentile(unique_lens, 66.67)
df["tertile"] = pd.cut(df["resp_len"], bins=[-np.inf, q33, q66, np.inf],
                       labels=["T1", "T2", "T3"])
df["origin"]  = df["response_source"].map({"AI":"AI","CoT AI":"AI","Human":"Human"})

EVALUATORS = [e for e in df["evaluator"].unique() if e != "Human"]
FORMULA    = "accuracy_score ~ C(tertile, Treatment('T1')) + C(origin, Treatment('Human'))"

# ── Fit and extract via summary2 ──────────────────────────────────────────────
records = []
for ev in EVALUATORS:
    sub = df[df["evaluator"] == ev].dropna(subset=["accuracy_score","tertile","origin"])
    mod = smf.ols(FORMULA, data=sub).fit()
    tbl = mod.summary2().tables[1]   # DataFrame: Coef, Std.Err., t, P>|t|, [0.025, 0.975]
    tbl["evaluator"] = ev
    tbl["term"]      = tbl.index
    records.append(tbl)

coef_df = pd.concat(records, ignore_index=True)
print(coef_df[["evaluator","term","Coef.","[0.025","0.975]","P>|t|"]].to_string(index=False))

# ── Forest plot ───────────────────────────────────────────────────────────────
TERMS = {
    "C(tertile, Treatment('T1'))[T.T2]": "T2 vs T1 (Medium vs Short)",
    "C(tertile, Treatment('T1'))[T.T3]": "T3 vs T1 (Long vs Short)",
    "C(origin, Treatment('Human'))[T.AI]": "AI vs Human",
}

FONT   = dict(family="Arial", color="black")
COLORS = ["#4393C3", "#2166AC", "#D6604D"]

ev_order = sorted(EVALUATORS)
y_pos    = list(range(len(ev_order)))

fig = make_subplots(rows=1, cols=len(TERMS),
                    subplot_titles=list(TERMS.values()),
                    horizontal_spacing=0.12)

for col_i, (term, title, color) in enumerate(zip(TERMS, TERMS.values(), COLORS), 1):
    sub  = coef_df[coef_df["term"] == term].set_index("evaluator")
    coef = [sub.loc[ev, "Coef."]   if ev in sub.index else np.nan for ev in ev_order]
    lo   = [sub.loc[ev, "[0.025"]  if ev in sub.index else np.nan for ev in ev_order]
    hi   = [sub.loc[ev, "0.975]"]  if ev in sub.index else np.nan for ev in ev_order]
    pv   = [sub.loc[ev, "P>|t|"]   if ev in sub.index else np.nan for ev in ev_order]

    fig.add_vline(x=0, line=dict(color="#aaaaaa", width=1, dash="dash"), row=1, col=col_i)

    fig.add_trace(go.Scatter(
        x=coef, y=y_pos, mode="markers",
        marker=dict(
            color=[color if p < 0.05 else "#cccccc" for p in pv],
            size=10, line=dict(color="black", width=0.8),
        ),
        error_x=dict(
            type="data", symmetric=False,
            array=[h - c for h, c in zip(hi, coef)],
            arrayminus=[c - l for c, l in zip(coef, lo)],
            color="#444444", thickness=1.4, width=5,
        ),
        showlegend=False,
    ), row=1, col=col_i)

    xl = f"xaxis{col_i if col_i > 1 else ''}"
    yl = f"yaxis{col_i if col_i > 1 else ''}"
    fig.update_layout(**{
        xl: dict(title=dict(text="Δ accuracy score", font=dict(**FONT, size=11)),
                 tickfont=dict(**FONT, size=10), showgrid=False, zeroline=False,
                 showline=True, linewidth=1, linecolor="black",
                 mirror=False, ticks="outside", ticklen=3),
        yl: dict(tickvals=y_pos, ticktext=ev_order if col_i == 1 else [""] * len(ev_order),
                 tickfont=dict(**FONT, size=10), showgrid=False, zeroline=False,
                 showline=True if col_i == 1 else False, linewidth=1, linecolor="black",
                 mirror=False),
    })

fig.update_layout(
    template="plotly_white", paper_bgcolor="white", plot_bgcolor="white",
    font=dict(**FONT, size=11),
    title=dict(text="OLS: accuracy ~ tertile + origin  (● p<0.05, ● n.s.)",
               font=dict(**FONT, size=13), x=0.5, xanchor="center"),
    width=900, height=320,
    margin=dict(t=60, b=50, l=120, r=20),
)

fig.write_image("ols_accuracy_tertile_origin.svg", scale=3)
fig.write_image("ols_accuracy_tertile_origin.png", scale=3)
fig.show()


In [ ]:
import re
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from IPython.display import display
from src.config import load_config
from src.data import load_cardio_dataset

cfg = load_config("../../../config/config.yaml")
raw = load_cardio_dataset(cfg)
df  = raw.to_pandas()

# ── IDK filter ────────────────────────────────────────────────────────────────
_IDK_RE = re.compile(
    r"(i\s+don'?t\s+know"
    r"|i\s+do\s+not\s+know"
    r"|i'?m\s+not\s+sure"
    r"|i\s+cannot\s+(answer|provide|help)"
    r"|i\s+can'?t\s+(answer|provide|help)"
    r"|no\s+(information|data)\s+available"
    r"|unable\s+to\s+(provide|answer)"
    r")",
    re.IGNORECASE,
)
_REF_RE = re.compile(r"\n\n?references?:.*", re.IGNORECASE | re.DOTALL)

df = df[df["response"].apply(
    lambda t: not bool(_IDK_RE.search(_REF_RE.sub("", t).strip()))
)].copy()
print(f"Rows after IDK filter: {len(df)}")

# ── Tertiles ──────────────────────────────────────────────────────────────────
df["resp_len"]  = df["response"].str.split().str.len()
unique_lens     = df.drop_duplicates("response")["resp_len"].values
q33 = np.percentile(unique_lens, 33.33)
q66 = np.percentile(unique_lens, 66.67)
df["tertile"]   = pd.cut(df["resp_len"], bins=[-np.inf, q33, q66, np.inf],
                         labels=["T1", "T2", "T3"])
df["origin"]    = df["response_source"].map({"AI":"AI","CoT AI":"AI","Human":"Human"})

# ── OLS per evaluator ─────────────────────────────────────────────────────────
EVALUATORS = sorted(e for e in df["evaluator"].unique())
FORMULA    = "accuracy_score ~ C(tertile, Treatment('T1')) * C(origin, Treatment('Human'))"

for ev in EVALUATORS:
    sub = df[df["evaluator"] == ev].dropna(subset=["accuracy_score", "tertile", "origin"])

    print(df["evaluator"].value_counts())
    mod = smf.ols(FORMULA, data=sub).fit()
    print(f"\n{'='*60}")
    print(f"  {ev}  (n={len(sub)})")
    print(f"{'='*60}")
    display(mod.summary())



### 8.4  Export shuffle plots (SVG)

In [ ]:
EXPORT_DIR = ROOT / "results" / "elastic_plots"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for metric in ["most_likely", "expected_score"]:
    fig = shuffle_violin_plot(metric=metric, groups=["Human", "AI"], height=700, width=1200)
    out = EXPORT_DIR / f"{MODEL_KEY}_shuffle_violin_{metric}.svg"
    fig.write_image(str(out), scale=3)
    print(f"Saved: {out.name}")

In [ ]:
import pickle, json
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT      = Path("../../../")
PROBE_DIR = ROOT / "results/probes"
STEER_DIR = ROOT / "results/steering"
TABLE_DIR = ROOT / "results/tables"
MODEL_KEY = "llama-3.1-8b"
FONT      = dict(family="Arial", color="black")

def _axis(title="", **kw):
    return dict(
        title=dict(text=title, font=dict(**FONT, size=16)),
        tickfont=dict(**FONT, size=16),
        showgrid=False, zeroline=False,
        showline=True, linewidth=1, linecolor="black",
        mirror=False, ticks="outside", ticklen=4, tickwidth=1,
        **kw
    )

# ── Plot 1: AUC por capa ──────────────────────────────────────────────────────
PROBES_AUC = [
    ("authorship",   "Authorship<br>(AI vs Human)",   "#2166AC"),
    ("accuracy",     "Accuracy<br>(high vs low)",      "#D6604D"),
    ("completeness", "Completeness<br>(high vs low)",  "#762A83"),
    ("clarity",      "Clarity<br>(high vs low)",       "#1B7837"),
    ("verbosity",    "Verbosity<br>(long vs short)",   "#E08A00"),
]

def load_kfold(probe_type, position="score_token"):
    r = pickle.load(open(PROBE_DIR / f"{MODEL_KEY}_{probe_type}_kfold_results.pkl", "rb"))
    rows = sorted([(x.layer, x.auc, x.auc_std) for x in r if x.position == position],
                  key=lambda x: x[0])
    return (np.array([x[0] for x in rows]),
            np.array([x[1] for x in rows]),
            np.array([x[2] for x in rows]))

fig1 = make_subplots(rows=1, cols=5,
                     subplot_titles=[p[1].replace("<br>", " ") for p in PROBES_AUC],
                     horizontal_spacing=0.06)

for ci, (probe, label, color) in enumerate(PROBES_AUC, 1):
    layers, aucs, stds = load_kfold(probe)
    peak_idx = int(np.argmax(aucs))
    xl = f"xaxis{ci if ci > 1 else ''}"
    yl = f"yaxis{ci if ci > 1 else ''}"

    fig1.add_hline(y=0.5, line=dict(color="#aaaaaa", width=0.8, dash="dash"), row=1, col=ci)

    fig1.add_trace(go.Scatter(
        x=np.concatenate([layers, layers[::-1]]),
        y=np.concatenate([aucs + stds, (aucs - stds)[::-1]]),
        fill="toself", fillcolor=color, opacity=0.15,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ), row=1, col=ci)

    fig1.add_trace(go.Scatter(
        x=layers, y=aucs, mode="lines",
        line=dict(color=color, width=2),
        name=label.replace("<br>", " "), showlegend=False,
    ), row=1, col=ci)

    fig1.add_trace(go.Scatter(
        x=[layers[peak_idx]], y=[aucs[peak_idx]], mode="markers+text",
        marker=dict(color=color, size=12, line=dict(color="white", width=1.5)),
        text=[f"L{layers[peak_idx]}  {aucs[peak_idx]:.2f}"],
        textposition="bottom right",
        textfont=dict(size=14, color=color, family="Arial"),
        showlegend=False,
    ), row=1, col=ci)

    fig1.add_vline(x=layers[peak_idx], line=dict(color=color, width=0.8, dash="dot"), row=1, col=ci)

    fig1.update_layout(**{
        xl: _axis("Layer", dtick=8, range=[-1, 33]),
        yl: _axis("AUROC (5-fold CV)" if ci == 1 else "", dtick=0.1, range=[0.42, 1.05]),
    })

fig1.update_layout(
    template="plotly_white",
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(**FONT, size=18),
    title=dict(text="Linear probe AUROC by layer — Llama 3.1 8B",
               font=dict(**FONT, size=18), x=0.5, xanchor="center"),
    width=1100, height=250,
    margin=dict(t=60, b=50, l=60, r=20),
)

for ax in [k for k in fig1.layout._props if k.startswith(("xaxis", "yaxis"))]:
    fig1.layout[ax].mirror = False



fig1.write_image("probe_auc_panel.svg", scale=3)
fig1.write_image("probe_auc_panel.png", scale=3)
fig1.show()


In [ ]:
import pickle
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

ROOT      = Path("../../../")
PROBE_DIR = ROOT / "results/probes"
MODEL_KEY = "llama-3.1-8b"
FONT      = dict(family="Arial", color="black")

PROBES_AUC = [
    ("authorship",   "Authorship (AI vs Human)",   "#2166AC"),
    ("accuracy",     "Accuracy (high vs low)",      "#D6604D"),
    ("completeness", "Completeness (high vs low)",  "#762A83"),
    ("clarity",      "Clarity (high vs low)",       "#1B7837"),
    ("verbosity",    "Verbosity (long vs short)",   "#E08A00"),
]

def load_kfold(probe_type, position="score_token"):
    r = pickle.load(open(PROBE_DIR / f"{MODEL_KEY}_{probe_type}_kfold_results.pkl", "rb"))
    rows = sorted([(x.layer, x.auc, x.auc_std) for x in r if x.position == position],
                  key=lambda x: x[0])
    return (np.array([x[0] for x in rows]),
            np.array([x[1] for x in rows]),
            np.array([x[2] for x in rows]))

fig = go.Figure()

# chance line
fig.add_hline(y=0.5, line=dict(color="#bbbbbb", width=0.8, dash="dash"))

for probe, label, color in PROBES_AUC:
    layers, aucs, stds = load_kfold(probe)
    peak_idx = int(np.argmax(aucs))

    # std band (no legend entry)
    fig.add_trace(go.Scatter(
        x=np.concatenate([layers, layers[::-1]]),
        y=np.concatenate([aucs + stds, (aucs - stds)[::-1]]),
        fill="toself", fillcolor=color, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ))

    # AUC line
    fig.add_trace(go.Scatter(
        x=layers, y=aucs, mode="lines",
        line=dict(color=color, width=2),
        name=label, legendgroup=probe,
    ))

    # peak marker
    fig.add_trace(go.Scatter(
        x=[layers[peak_idx]], y=[aucs[peak_idx]], mode="markers+text",
        marker=dict(color=color, size=7, line=dict(color="white", width=1.5)),
        text=[f"L{layers[peak_idx]} {aucs[peak_idx]:.2f}"],
        textposition="top right",
        textfont=dict(size=9, color=color, family="Arial"),
        showlegend=False, legendgroup=probe,
    ))

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(**FONT, size=12),
    title=dict(text="Linear probe AUROC by layer — Llama 3.1 8B",
               font=dict(**FONT, size=13), x=0.5, xanchor="center"),
    xaxis=dict(
        title=dict(text="Layer", font=dict(**FONT, size=12)),
        tickfont=dict(**FONT, size=11),
        showgrid=False, zeroline=False,
        showline=True, linewidth=1, linecolor="black",
        mirror=False, ticks="outside", ticklen=4, dtick=4, range=[-0.5, 32.5],
    ),
    yaxis=dict(
        title=dict(text="AUROC (5-fold CV)", font=dict(**FONT, size=12)),
        tickfont=dict(**FONT, size=11),
        showgrid=False, zeroline=False,
        showline=True, linewidth=1, linecolor="black",
        mirror=False, ticks="outside", ticklen=4, dtick=0.1, range=[0.42, 1.05],
    ),
    legend=dict(
        orientation="h", x=0.5, xanchor="center", y=-0.18, yanchor="top",
        bgcolor="rgba(0,0,0,0)", borderwidth=0,
        font=dict(**FONT, size=10),
    ),
    margin=dict(t=55, b=90, l=65, r=20),
    width=520, height=300,

)

fig.write_image("probe_auc_single.svg", scale=3)
fig.write_image("probe_auc_single.png", scale=3)
fig.show()

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT      = Path("../../../")
STEER_DIR = ROOT / "results/steering"
MODEL_KEY = "llama-3.1-8b"
FONT      = dict(family="Arial", color="black")

def _axis(title="", **kw):
    return dict(
        title=dict(text=title, font=dict(**FONT, size=14)),
        tickfont=dict(**FONT, size=14),
        showgrid=False, zeroline=False,
        showline=True, linewidth=1.5, linecolor="black",
        mirror=False, ticks="outside", ticklen=4, tickwidth=1,
        **kw
    )

# ── Plot 2: Steering asimétrico L15 — expected score ─────────────────────────
df_steer = pd.read_csv(STEER_DIR / f"{MODEL_KEY}_asym_steering.csv")

PROBES_STEER = [
    ("authorship", "Authorship direction", "#2166AC"),
    ("accuracy",   "Accuracy direction",   "#D6604D"),
    ("verbosity",  "Verbosity direction",  "#E08A00"),
]
LAYER  = 15
ALPHAS = [0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]

fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=[p[1] for p in PROBES_STEER],
                     horizontal_spacing=0.10)

for ci, (probe, title, color) in enumerate(PROBES_STEER, 1):
    sub = df_steer[
        (df_steer["probe_type"] == probe) &
        (df_steer["dir_kind"] == "logistic") &
        (df_steer["steer_layer"] == LAYER) &
        (df_steer["alpha"].isin(ALPHAS))
    ].sort_values("alpha")

    alphas = sub["alpha"].values
    ai     = sub["ai_mean_expected"].values
    human  = sub["human_mean_expected"].values
    bias   = ai - human
    xl = f"xaxis{ci if ci > 1 else ''}"
    yl = f"yaxis{ci if ci > 1 else ''}"

    # bias fill
    fig2.add_trace(go.Scatter(
        x=np.concatenate([alphas, alphas[::-1]]),
        y=np.concatenate([ai, human[::-1]]),
        fill="toself", fillcolor=color, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ), row=1, col=ci)

    # AI line
    fig2.add_trace(go.Scatter(
        x=alphas, y=ai, mode="lines+markers",
        line=dict(color=color, width=2),
        marker=dict(size=5, color=color),
        name="AI", showlegend=(ci == 1), legendgroup="AI",
    ), row=1, col=ci)

    # Human line
    fig2.add_trace(go.Scatter(
        x=alphas, y=human, mode="lines+markers",
        line=dict(color="#555555", width=2, dash="dash"),
        marker=dict(size=5, color="#555555", symbol="square"),
        name="Human", showlegend=(ci == 1), legendgroup="Human",
    ), row=1, col=ci)

    # crossover vline
    cross = np.where(np.diff(np.sign(bias)))[0]
    if len(cross):
        fig2.add_vline(x=alphas[cross[0] + 1],
                       line=dict(color=color, width=0.8, dash="dot"),
                       row=1, col=ci)

    fig2.update_layout(**{
        xl: _axis("α (steering strength)", dtick=2, range=[-0.3, 8.5]),
        yl: _axis("Expected score Σ d·P(d)" if ci == 1 else "",
                  dtick=2, range=[0.5, 7.0]),
    })

fig2.update_layout(
    template="plotly_white",
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(**FONT, size=21),
    title=dict(text=f"Asymmetric projection steering — L{LAYER} — Llama 3.1 8B",
               font=dict(**FONT, size=22), x=0.5, xanchor="center"),
    width=660, height=250,
    margin=dict(t=60, b=70, l=65, r=20),
    legend=dict(orientation="v", x=1.15, xanchor="center", y=0.6,
                font=dict(size=16, family="Arial"), bgcolor="rgba(0,0,0,0)"),
)

fig2.write_image("asym_steering_L15_expected.svg", scale=3)
fig2.write_image("asym_steering_L15_expected.png", scale=3)
fig2.show()


In [ ]:
cosine_data = json.load(open(TABLE_DIR / f"{MODEL_KEY}_layerwise_cosines_score_token.json"))

PAIRS = [
    ("authorship_vs_accuracy_probe",  "Authorship vs Accuracy",  "#9E3A26", "solid"),
    ("authorship_vs_verbosity_probe", "Authorship vs Verbosity", "#2166AC", "dash"),
    ("accuracy_vs_verbosity_probe",   "Accuracy vs Verbosity",   "#5AAE61", "dot"),
]
PEAK_LAYERS = {"authorship": 6, "accuracy": 18, "verbosity": 1}
PEAK_COLORS = {"authorship": "#2166AC", "accuracy": "#D6604D", "verbosity": "#E08A00"}

fig3 = go.Figure()

fig3.add_hline(y=0, line=dict(color="#cccccc", width=0.8))

for key, label, color, dash in PAIRS:
    entry   = cosine_data[key]
    layers  = np.array(entry["layers"])
    cosines = np.array(entry["cosines"])
    fig3.add_trace(go.Scatter(
        x=layers, y=cosines, mode="lines",
        line=dict(color=color, width=2, dash=dash),
        name=label,
    ))

for probe, layer in PEAK_LAYERS.items():
    fig3.add_vline(x=layer,
                   line=dict(color=PEAK_COLORS[probe], width=0.8, dash="dot"),
                   opacity=0.55)
    fig3.add_annotation(
        x=layer, y=0.205, text=f"L{layer}",
        showarrow=False, font=dict(size=19, color=PEAK_COLORS[probe], family="Arial"),
        xanchor="left"
    )

fig3.update_layout(
    template="plotly_white",
    paper_bgcolor="white", plot_bgcolor="white",
    width=520, height=340,
    font=dict(family="Arial", color="black", size=11),
    title=dict(text="Probe direction geometry — Llama 3.1 8B",
               font=dict(size=22, family="Arial"), x=0.5, xanchor="center"),
    xaxis=_axis("Layer", dtick=8, range=[-1, 33]),
    yaxis=_axis("Cosine similarity", dtick=0.05, range=[-0.02, 0.23]),
    legend=dict(x=0.7, y=1.12, bgcolor="rgba(0,0,0,0)",
                font=dict(size=16, family="Arial"), borderwidth=0),
    margin=dict(t=50, b=50, l=65, r=20),
)

fig3.write_image("probe_cosines.svg", scale=3)
fig3.write_image("probe_cosines.png", scale=3)
fig3.show()
